# HTG cases: cellulose and Yakaboylu Figures 7–8
Run this notebook from top to bottom in a fresh kernel. Keep `HTG_Model(2).ipynb` beside it, together with your four databases and `Cellulose.xlsx` (or set explicit paths below).

Only two case families are included. The model notebook supplies the thermodynamic functions and solver; no previously executed cell or Jupyter history is required.

**Figures 7–8 retained version:** common NASA H2O reference, liquid-rich initial water below 390 °C, aqueous phase excluded at and above 390 °C. This is the reduced C/H/O/N/S comparison, not a full-manure reproduction.

In [ ]:
from pathlib import Path
import json
import types
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_NOTEBOOK = Path('HTG_Model(2).ipynb')  # Updated model saved alongside this notebook.
DATABASE_DIRECTORY = Path('.')


def locate_file(preferred_names):
    for folder in (DATABASE_DIRECTORY, DATABASE_DIRECTORY / 'upload'):
        for name in preferred_names:
            candidate = folder / name
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f'File not found; set an explicit path: {preferred_names}')


DATABASE_PATHS = {
    'nasa': locate_file(['Thermdat_WasteMech.txt']),
    'pcsaft': locate_file(['PROJECT_PCSAFT.csv']),
    'epcsaft': locate_file(['PROJECT_EPCSAFT.csv']),
    'supcrt': locate_file(['supcrt92(7).xml', 'supcrt92.xml']),
}


def load_model_notebook(path):
    document = json.loads(Path(path).read_text(encoding='utf-8'))
    module = types.ModuleType('htg_model')
    for index, cell in enumerate(document['cells']):
        if cell['cell_type'] == 'code':
            text = ''.join(cell['source'])
            exec(compile(text, f'{path}:cell{index + 1}', 'exec'), module.__dict__)
    return module


model = load_model_notebook(MODEL_NOTEBOOK)
SPECIES_OVERRIDES = {}  # e.g. {'ethanol': {'nasa': 'C2H5OH'}}
available = model.initialize_model(DATABASE_PATHS, species_overrides=SPECIES_OVERRIDES)
# Full inventory, including missing EOS/reference coverage:
species_catalogue = model.available_species(include_unavailable=True)
display(available)


## 1. Cellulose settings
The default feed and former MATLAB results come from `Cellulose.xlsx`. To override the dry feed, set either `feed_element_wt_percent` or `feed_species_moles` (molar dry-feed composition). Added water is separate. `None` keeps the workbook value for feed or water.

Edit the phase species lists to include/exclude database-covered products. The original cellulose comparison uses the **molecular aqueous model**, unlike the electrolyte model used for Figures 7–8. Temperature and pressure are editable; the former-result markers remain at the workbook's original states.

`gas_carbon_fraction=0.9` preserves the previous cellulose constraint. Set it to `None` for unconstrained gas-carbon allocation. Fractions refer to total feed carbon; they are not imposed reaction rates. Optional initial mole amounts use `(phase, species)` keys and units mol/mol dry-feed C; they must satisfy all balances.
Use the database-discovered species names displayed in `available`. To inspect unavailable records, display `species_catalogue`; ambiguous NASA mappings can be added to `SPECIES_OVERRIDES` in the loading cell.


In [ ]:
CELLULOSE_WORKBOOK = locate_file(['Cellulose.xlsx'])
CELLULOSE_SETTINGS = {
    'temperatures_C': [375, 400, 425, 450, 475, 500, 525, 550],
    'pressure_bar': 275.0,
    'feed_element_wt_percent': None,
    'feed_species_moles': None,             # e.g. {'cellulose': 1.0}
    'water_g_per_g_dry_feed': None,         # None uses the workbook feed ratio
    'added_salts': {},
    'phase_species': {
    'gas': [
        'water', 'hydrogen', 'carbon dioxide',
        'carbon monoxide', 'methane', 'methanol',
        'ammonia', 'hydrogen sulfide',
    ],
    'aqueous': [
        'water', 'hydrogen', 'carbon dioxide',
        'carbon monoxide', 'methane', 'methanol',
        'ammonia', 'hydrogen sulfide',
    ],
    'organic liquid': [
        'benzene', 'toluene',
    ],
    'solid': [
        'carbon', 'cellulose', 'Lignin1',
    ],
},
    'gas_carbon_fraction': 0.9,
    'solid_carbon_fraction': None,
    'solid_carbon_target': 'CHAR',
    'minimum_amounts': {('gas', 'water'): 1e-8},
    'initial_phase_amounts': None,
    'aqueous_model': 'molecular',
    'aqueous_water_floor': 0.0,
    'liquid_initialization': False,
    'gas_water_initialization': True,
    'optimizer_ftol': 1e-11,
    'max_iterations': 1200,
}


In [ ]:
# Read the workbook with the same cell addresses as the original comparison.
input_sheet = pd.read_excel(CELLULOSE_WORKBOOK, sheet_name='Input', header=None)
cellulose_workbook_feed = {
    str(input_sheet.iat[row, 0]).strip(): float(input_sheet.iat[row, 1])
    for row in range(6, 11)
}
workbook_atomic_mass = {
    str(input_sheet.iat[row, 0]).strip(): float(input_sheet.iat[row, 2])
    for row in range(6, 11)
}
solid_feed_kg = float(input_sheet.iat[7, 6])
total_feed_kg = float(input_sheet.iat[6, 6])
water_feed_kg = total_feed_kg - solid_feed_kg
if solid_feed_kg <= 0 or water_feed_kg < 0:
    raise ValueError('Invalid cellulose workbook feed mass')
if not np.isclose(sum(cellulose_workbook_feed.values()), 100.0):
    raise ValueError('Workbook dry-feed wt% does not sum to 100')
carbon_kmol = solid_feed_kg * cellulose_workbook_feed['C'] / 100 / workbook_atomic_mass['C']
if carbon_kmol <= 0:
    raise ValueError('Workbook contains no dry-feed carbon')
# Preserve the original workbook normalization and water molecular weight.
water_per_mol_C = (water_feed_kg / 18.01528) / carbon_kmol
# Use workbook atomic masses for this comparison, as in the original notebook.
cellulose_atomic_weights = dict(workbook_atomic_mass)
old_to_new = {'H2O(g)': 'water', 'CO2(g)': 'carbon dioxide',
              'CH4(g)': 'methane', 'H2(g)': 'hydrogen', 'CO(g)': 'carbon monoxide',
              'NH3(g)': 'ammonia', 'H2S(g)': 'hydrogen sulfide'}
cellulose_reference_rows = []
for sheet_name in ('400 C, 275 bar', '500 C, 275 bar'):
    sheet = pd.read_excel(CELLULOSE_WORKBOOK, sheet_name=sheet_name, header=None)
    amounts = {str(sheet.iat[row, 0]).strip(): float(sheet.iat[row, 1])
               for row in range(4, 17)
               if pd.notna(sheet.iat[row, 0]) and pd.notna(sheet.iat[row, 1])}
    row = {'Temperature [°C]': float(sheet.iat[0, 1]),
           'Pressure [bar]': float(sheet.iat[1, 1]),
           'Dry gas [mol/mol feed C]': float(sheet.iat[17, 1]) / carbon_kmol}
    row.update({name: amounts.get(old, 0.0) / carbon_kmol
                for old, name in old_to_new.items()})
    cellulose_reference_rows.append(row)
cellulose_reference_results = pd.DataFrame(cellulose_reference_rows)

cellulose_rows, cellulose_species_rows, cellulose_case_results = [], [], []
# Keep the workbook's atom ratios exactly unless you supply a replacement feed.
saved_AW = dict(model.AW)
try:
    if CELLULOSE_SETTINGS['feed_element_wt_percent'] is None and CELLULOSE_SETTINGS['feed_species_moles'] is None:
        model.AW = {**model.AW, **cellulose_atomic_weights}
    for temperature_C in CELLULOSE_SETTINGS['temperatures_C']:
        settings = copy.deepcopy(CELLULOSE_SETTINGS)
        settings['temperature_C'] = temperature_C
        if settings['feed_element_wt_percent'] is None:
            settings['feed_element_wt_percent'] = dict(cellulose_workbook_feed)
        if settings['water_g_per_g_dry_feed'] is None:
            settings['water_g_per_g_dry_feed'] = water_feed_kg / solid_feed_kg
            if CELLULOSE_SETTINGS['feed_element_wt_percent'] is None and CELLULOSE_SETTINGS['feed_species_moles'] is None:
                # Same added-water mol/mol C as the original workbook comparison.
                c_per_g = settings['feed_element_wt_percent']['C'] / 100 / model.AW['C']
                settings['water_g_per_g_dry_feed'] = water_per_mol_C * c_per_g * (2 * model.AW['H'] + model.AW['O'])
        row = {'Temperature [°C]': float(temperature_C), 'Pressure [bar]': settings['pressure_bar']}
        row.update({s: np.nan for s in old_to_new.values()})
        row['Dry gas [mol/mol feed C]'] = np.nan
        try:
            result = model.solve_equilibrium(settings)
            cellulose_case_results.append(result)
            gas = {s: amount for (phase, s), amount in result['amounts'].items() if phase == 'gas'}
            row.update({s: gas.get(s, 0.0) for s in old_to_new.values()})
            row.update(Status='solved', Error='', Balance_residual=result['balance_residual'])
            row['Dry gas [mol/mol feed C]'] = sum(amount for s, amount in gas.items() if s != 'water')
            for (phase, species), amount in result['amounts'].items():
                cellulose_species_rows.append({'Temperature [°C]': temperature_C, 'Phase': phase,
                    'Species': species, 'Amount [mol/mol feed C]': amount})
        except Exception as exc:
            row.update(Status='failed', Error=str(exc), Balance_residual=np.nan)
        cellulose_rows.append(row)
        print(f'{temperature_C:g} °C: {row["Status"]}')
finally:
    model.AW = saved_AW
cellulose_results = pd.DataFrame(cellulose_rows)
cellulose_species_results = pd.DataFrame(cellulose_species_rows)
with pd.option_context('display.max_colwidth', None):
    display(cellulose_results)
display(cellulose_reference_results)
if not cellulose_species_results.empty:
    display(cellulose_species_results)


In [ ]:
cellulose_plot_species = {'carbon dioxide':'CO₂', 'methane':'CH₄',
                          'hydrogen':'H₂', 'carbon monoxide':'CO'}
fig, axes = plt.subplots(3, 2, figsize=(12, 10), layout='constrained')
for ax, (species, label) in zip(axes.ravel(), [*cellulose_plot_species.items(),
        ('Dry gas [mol/mol feed C]', 'Total dry gas')]):
    ax.plot(cellulose_results['Temperature [°C]'], cellulose_results[species],
            'o-', label='Current model')
    ax.scatter(cellulose_reference_results['Temperature [°C]'],
               cellulose_reference_results[species], marker='s', label='Former MATLAB')
    ax.set(title=label, xlabel='Temperature [°C]', ylabel='mol/mol dry-feed C')
    ax.grid(alpha=.25); ax.legend()
axes.ravel()[-1].axis('off')
fig.suptitle(f'Cellulose comparison — current model at {CELLULOSE_SETTINGS["pressure_bar"]:g} bar')
plt.show()


In [ ]:
# CELLULOSE: PHASE DISTRIBUTION AND COMPOSITION OF EACH PHASE

phase_order = ['gas', 'aqueous', 'organic liquid', 'solid']
temperatures = pd.Index(
    CELLULOSE_SETTINGS['temperatures_C'],
    name='Temperature [°C]'
)

phase_amounts = pd.DataFrame(
    np.nan, index=temperatures, columns=phase_order
)
phase_mass = phase_amounts.copy()
phase_carbon = phase_amounts.copy()

phase_compositions = {
    phase: pd.DataFrame(
        np.nan,
        index=temperatures,
        columns=CELLULOSE_SETTINGS['phase_species'][phase]
    )
    for phase in phase_order
}

case_atomic_weights = {
    **model.AW,
    **CELLULOSE_SETTINGS.get('atomic_weights', {})
}

for result in cellulose_case_results:
    temperature = result['temperature_C']

    for phase in phase_order:
        selected = CELLULOSE_SETTINGS['phase_species'][phase]
        amounts = np.array([
            result['amounts'].get((phase, species), 0.0)
            for species in selected
        ], dtype=float)

        registry = (
            model.SOLID_DATA
            if phase == 'solid'
            else {
                **model.MOLECULAR_SPECIES,
                **model.ORGANIC_SPECIES,
                **model.ION_DATA
            }
        )

        molecular_weights = np.array([
            sum(
                count * case_atomic_weights[element]
                for element, count
                in registry[species]['elements'].items()
            )
            for species in selected
        ], dtype=float)

        carbon_counts = np.array([
            registry[species]['elements'].get('C', 0.0)
            for species in selected
        ], dtype=float)

        phase_amounts.loc[temperature, phase] = amounts.sum()
        phase_mass.loc[temperature, phase] = amounts @ molecular_weights
        phase_carbon.loc[temperature, phase] = amounts @ carbon_counts

        # Composition is undefined when the phase is absent.
        if amounts.sum() > 1e-12:
            phase_compositions[phase].loc[temperature] = (
                amounts / amounts.sum()
            )

# Mass fractions include all products, including added water.
phase_mass_fractions = phase_mass.div(
    phase_mass.sum(axis=1, min_count=1),
    axis=0
)

# Actual feed-carbon balance also accounts for added salts.
feed_carbon = pd.Series(np.nan, index=temperatures)
for result in cellulose_case_results:
    feed_carbon.loc[result['temperature_C']] = (
        result['feed_atoms'][result['elements'].index('C')]
    )

phase_carbon_fractions = phase_carbon.div(feed_carbon, axis=0)

# ------------------------------------------------------------
# 1. Phase amounts, mass distribution, and carbon allocation
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 3, figsize=(17, 5), layout='constrained'
)

plot_data = [
    (
        phase_amounts,
        'Total phase amounts',
        'mol/mol dry-feed C'
    ),
    (
        phase_mass_fractions,
        'Product mass distribution (including water)',
        'Mass fraction of all products'
    ),
    (
        phase_carbon_fractions,
        'Feed-carbon allocation',
        'Fraction of total feed carbon'
    )
]

for ax, (values, title, ylabel) in zip(axes, plot_data):
    for phase in phase_order:
        ax.plot(values.index, values[phase], 'o-', label=phase)

    ax.set(
        title=title,
        xlabel='Temperature [°C]',
        ylabel=ylabel,
        xlim=(float(temperatures.min()), float(temperatures.max()))
    )
    if values is not phase_amounts:
        ax.set_ylim(0, 1.05)

    ax.grid(alpha=.25)
    ax.legend()

fig.suptitle(
    f'Cellulose phase distribution — '
    f'{CELLULOSE_SETTINGS["pressure_bar"]:g} bar'
)
plt.show()

# ------------------------------------------------------------
# 2. Species mole fractions within each phase, including water
#    Solid fractions refer to species formula units.
# ------------------------------------------------------------
fig, axes = plt.subplots(
    2, 2, figsize=(14, 10), layout='constrained'
)

for ax, phase in zip(axes.ravel(), phase_order):
    composition = phase_compositions[phase]

    for species in composition.columns:
        ax.plot(
            composition.index,
            composition[species],
            'o-',
            ms=3,
            label=species
        )

    ax.set(
        title=phase,
        xlabel='Temperature [°C]',
        ylabel='Species mole fraction in phase',
        ylim=(0, 1.05),
        xlim=(float(temperatures.min()), float(temperatures.max()))
    )
    ax.grid(alpha=.25)

    if len(composition.columns):
        ax.legend(fontsize=8)

    if not composition.notna().any().any():
        ax.text(
            .5, .5,
            'Phase absent or no solved states',
            ha='center',
            va='center',
            transform=ax.transAxes
        )

fig.suptitle(
    f'Cellulose phase compositions — '
    f'{CELLULOSE_SETTINGS["pressure_bar"]:g} bar'
)
plt.show()

## 2. Yakaboylu et al. (2013), Figures 7–8 settings
Retained successful configuration: reduced C/H/O/N/S feed, char as the only solid product, gas-only N₂, common NASA H₂O reference, liquid-rich starting composition below the cutoff, and aqueous excluded at/above 390 °C. Carbon allocation is unconstrained by default.

Edit temperatures, pressure, dry-feed elemental wt% (or molar species composition), added-water mass ratio, species lists, and gas/solid carbon fractions below. The published markers stay at their original 240 bar feed conditions. The reference curves are approximate digitizations of calculated equilibrium results, not experiments.

The aqueous model retains its density/permittivity validity checks. An aqueous solvent floor remains when that phase is offered. The fixed cutoff is a modelling assumption, not a predicted phase boundary. Failed cases remain NaN.

In [ ]:
FIG78_SETTINGS = {
    'temperatures_C': [100, 150, 200, 250, 300, 340, 360, 380, 389.6,
                       390, 400, 420, 460, 500, 540, 580],
    'pressure_bar': 240.0,
    'aqueous_cutoff_C': 390.0,
    'feed_element_wt_percent': {
        e: n * model.AW[e] / 25000 * 100
        for e, n in dict(C=632.65, H=1093.28, O=837.22, N=47.56, S=3.95).items()
    },
    'feed_species_moles': None,
    'water_g_per_g_dry_feed': 5550.84 * (2 * model.AW['H'] + model.AW['O']) / 25000,
    'added_salts': {},
    'phase_species': {
    'gas': [
        'water', 'hydrogen', 'carbon dioxide',
        'carbon monoxide', 'methane', 'methanol',
        'ammonia', 'hydrogen sulfide', 'nitrogen',
    ],
    'aqueous': [
        'water', 'hydrogen', 'carbon dioxide',
        'carbon monoxide', 'methane', 'methanol',
        'ammonia', 'hydrogen sulfide', 
    ],
    #if nitrogen is included in aqueous the dip is reproduced, solver wrong convergence for 380C !!!
    'organic liquid': [
        'benzene', 'toluene',
    ],
    'solid': [
        'carbon', 
    ],
},
    'gas_carbon_fraction': None,
    'solid_carbon_fraction': None,
    'solid_carbon_target': 'CHAR',
    'aqueous_model': 'electrolyte',
    'aqueous_water_floor': 1e-8,
    'liquid_initialization': True,
    'initial_gas_water_fraction': 1e-6,
    'initial_phase_amounts': None,
    'minimum_amounts': {},
    'optimizer_ftol': 1e-11,
    'optimizer_eps': 1e-8,
    'max_iterations': 10000,
}



In [ ]:
TC = np.array([100,150,200,250,300,340,360,380,389.6,390,400,420,460,500,540,580], dtype=float)
paper78 = pd.DataFrame({
 'T_C': TC,
 'CO2': [1.3,2.2,2.8,3.2,4.3,14.4,14.7,15.0,15.0,15.0,15.0,15.1,15.4,15.8,16.2,16.7],
 'CH4': [0,0,0,0,.2,9.2,9.5,9.9,9.9,9.9,9.9,9.8,9.5,9.1,8.6,8.0],
 'H2': [0,0,0,0,0,0,.3,1.1,1.2,1.2,1.3,1.8,2.9,4.3,6.4,8.9],
 'N2': [.75,.78,.75,.70,.70,.89,.91,.93,.93,.93,.93,.93,.93,.92,.92,.91],
 'NH3': [0,0,0,0,0,0,.03,.04,.04,.04,.04,.05,.05,.06,.06,.07],
 'H2S': [0,0,0,0,.05,.03,.02,.06,.16,.16,.16,.16,.16,.16,.16,.16],
 'CO': [0,0,0,0,0,0,0,0,0,0,.01,.01,.04,.08,.16,.28],
 'H2O_g': [0,0,0,1,5,55,142,218,223,223,223,222,222,221,220,219],
 'H2O_l': [242,242,241,240,238,167,106,5,0,0,0,0,0,0,0,0],
})
paper78['overlapping_marker'] = paper78.T_C.isin([389.6,390])


In [ ]:
fig78_rows, fig78_species_rows, fig78_initial_rows, fig78_case_results = [], [], [], []
fig78_map = {'CO2':'carbon dioxide', 'CH4':'methane', 'H2':'hydrogen',
             'N2':'nitrogen', 'NH3':'ammonia', 'H2S':'hydrogen sulfide', 'CO':'carbon monoxide'}
for temperature_C in FIG78_SETTINGS['temperatures_C']:
    settings = copy.deepcopy(FIG78_SETTINGS)
    settings['temperature_C'] = float(temperature_C)
    # The model applies the shared aqueous_cutoff_C rule before EOS setup.
    row = {'T_C':float(temperature_C), 'P_bar':settings['pressure_bar'], 'Liquid_water_NASA':'H2O'}
    row.update({column:np.nan for column in [*fig78_map, 'H2O_g', 'H2O_l']})
    try:
        result = model.solve_equilibrium(settings)
        fig78_case_results.append(result)
        # Return units are mol/mol dry-feed C. Convert to mol/kg dry feed.
        factor = result['feed_C_mol_per_g_dry'] * 1000
        yields = {pair: amount * factor for pair, amount in result['amounts'].items()}
        for column, species in fig78_map.items():
            row[column] = yields.get(('gas', species), 0.0)
        row['H2O_g'] = yields.get(('gas', 'water'), 0.0)
        row['H2O_l'] = sum(yields.get((phase, 'water'), 0.0)
                           for phase in ('aqueous', 'organic liquid'))
        row.update(Status='solved', Error='', Balance_residual=result['balance_residual'])
        initial = {'T_C':float(temperature_C)}
        for phase in ('gas', 'aqueous', 'organic liquid'):
            initial['Initial_water_' + phase.replace(' ', '_')] = result['initial_amounts'].get((phase, 'water'), 0.0)
        fig78_initial_rows.append(initial)
        for (phase, species), amount in yields.items():
            fig78_species_rows.append({'T_C':float(temperature_C), 'Phase':phase,
                                      'Species':species, 'Yield_mol_per_kg':amount})
    except Exception as exc:
        row.update(Status='failed', Error=str(exc), Balance_residual=np.nan)
    fig78_rows.append(row)
    print(f'{temperature_C:g} °C: {row["Status"]}')
fig78_model_results = pd.DataFrame(fig78_rows)
fig78_model_species = pd.DataFrame(fig78_species_rows)
fig78_initial_water = pd.DataFrame(fig78_initial_rows)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(fig78_model_results)
    display(fig78_initial_water)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout='constrained')
for ax, columns, title in zip(axes, [['CO2','CH4','H2'], ['N2','NH3','H2S','CO'], ['H2O_g','H2O_l']],
        ['Figure 7: major gases', 'Figure 7: minor gases', 'Figure 8: water']):
    for column in columns:
        line, = ax.plot(paper78.T_C, paper78[column], 'o', ms=4, label=f'{column}: paper')
        ax.plot(fig78_model_results.T_C, fig78_model_results[column], '-s', ms=3,
                color=line.get_color(), label=f'{column}: reduced model')
    ax.axvline(FIG78_SETTINGS['aqueous_cutoff_C'], color='0.5', ls='--', lw=.8)
    ax.set(title=title, xlabel='Temperature [°C]', ylabel='mol/kg dry manure', xlim=(100,580))
    ax.grid(alpha=.25); ax.legend(fontsize=7)
fig.suptitle(f'Figures 7–8 — reduced C/H/O/N/S feed at {FIG78_SETTINGS["pressure_bar"]:g} bar\n'
             'Common NASA H2O; liquid-rich start; aqueous excluded above the configured cutoff')
plt.show()


3. Original four-phase cellulose demonstration
The original example and its Gibbs functions are reproduced below, with the original feasibility LP, full equality matrix, objective scale 1e4, and SLSQP settings (maxiter 1200, ftol 1e-10, eps 1e-7). Feed: 1/6 mol cellulose, 10 mol water and 0.05 mol NaHCO3 at 350 °C and 250 bar. Prescribed carbon fractions: gas 0.55, aqueous 0.20, organic liquid 0.10, solid 0.15.
The wrapper initializes model references at the specified state and restores the model afterwards. This case calls the EOS/Gibbs functions directly and bypasses the generalized solve_equilibrium implementation. The following cell reproduces the original species plots. Native EOS convergence still requires execution in your environment.

In [ ]:
# Exact original four-phase calculation, isolated from other case results.
# Model functions are called directly; solve_equilibrium is not used.
FOUR_PHASE_ORIGINAL_SPECIES = {
    'gas': ['water', 'hydrogen', 'carbon dioxide', 'carbon monoxide',
            'methane', 'methanol', 'ammonia', 'hydrogen sulfide'],
    'aqueous': ['water', 'hydrogen', 'carbon dioxide', 'carbon monoxide',
                'methane', 'methanol', 'ammonia', 'hydrogen sulfide',
                'Na+', 'HCO3-', 'CO3^2-'],
    'organic liquid': ['benzene', 'toluene'],
    'solid': ['carbon', 'cellulose', 'Lignin1'],
}


def run_original_four_phase_example():
    import numpy as np
    import pandas as pd
    saved_model_state = dict(model.__dict__)
    try:
        model.configure_aqueous_species(FOUR_PHASE_ORIGINAL_SPECIES)
        model._refresh_state(350.0, 250.0, electrolyte=True,
                             phase_species=FOUR_PHASE_ORIGINAL_SPECIES)
        T, P = model.T, model.P
        cl, R, P0 = model.cl, model.R, model.P0
        build_pcsaft_model = model.build_pcsaft_model
        GAS_SPECIES = FOUR_PHASE_ORIGINAL_SPECIES['gas']
        AQ_NEUTRAL_SPECIES = FOUR_PHASE_ORIGINAL_SPECIES['aqueous'][:-3]
        AQ_ION_SPECIES = FOUR_PHASE_ORIGINAL_SPECIES['aqueous'][-3:]
        AQUEOUS_SPECIES = AQ_NEUTRAL_SPECIES + AQ_ION_SPECIES
        ORG_SPECIES = FOUR_PHASE_ORIGINAL_SPECIES['organic liquid']
        SOLID_SPECIES = FOUR_PHASE_ORIGINAL_SPECIES['solid']
        aqueous_epcsaft_model = model.aqueous_epcsaft_model
        mu0_ion = model.mu0_ion
        lnphi_infinity = model.lnphi_infinity
        mu0_gas, _ = model.nasa_standard_chemical_potentials(GAS_SPECIES)
        mu0_aq_neutral, _ = model.nasa_standard_chemical_potentials(AQ_NEUTRAL_SPECIES)
        mu0_org, _ = model.nasa_standard_chemical_potentials(ORG_SPECIES)
        mu0_solid, _ = model.solid_standard_chemical_potentials(SOLID_SPECIES)
        MW_WATER_KG = 0.01801528
        PHASES = [(phase, species) for phase, names in FOUR_PHASE_ORIGINAL_SPECIES.items()
                  for species in names]
        PHASE_INDEX = {phase: np.array([j for j, (p, _) in enumerate(PHASES) if p == phase], dtype=int)
                       for phase in FOUR_PHASE_ORIGINAL_SPECIES}
        ELEMENTS = ['C', 'H', 'O', 'N', 'S', 'Na']
        def species_data(phase, species):
            if phase == 'solid':
                return model.SOLID_DATA[species]
            if phase == 'aqueous' and species in model.ION_DATA:
                return model.ION_DATA[species]
            return {**model.MOLECULAR_SPECIES, **model.ORGANIC_SPECIES}[species]
        A_original = np.array([
            [species_data(phase, species)['elements'].get(element, 0.0)
             for phase, species in PHASES] for element in ELEMENTS
        ] + [[species_data(phase, species).get('charge', 0) if phase == 'aqueous' else 0
              for phase, species in PHASES]], dtype=float)

        def phi_electrolyte(n):
            n = np.asarray(n, dtype=float).copy()
            if n.shape != (len(AQUEOUS_SPECIES),): raise ValueError("Aqueous vector dimension")
            if np.any(n < 0): raise ValueError("Negative aqueous amount")
            # SLSQP probes individual coordinates while estimating derivatives.
            # Project ONLY the EOS probe onto an electroneutral composition.
            imbalance = float(np.dot(n[-3:], [1.,-1.,-2.]))
            smoothing = np.hypot(imbalance,2e-14)
            n[-3] += (smoothing-imbalance)/2
            n[-2] += (smoothing+imbalance)/2
            n[:len(AQ_NEUTRAL_SPECIES)] += 1e-14
            n[len(AQ_NEUTRAL_SPECIES):] += np.array([3.,1.,1.])*1e-14
            value = np.asarray(cl.fugacity_coefficient(aqueous_epcsaft_model,P,T,n,phase="liquid"),dtype=float).reshape(-1)
            if len(value)!=len(n) or np.any(~np.isfinite(value)) or np.any(value<=0):
                raise ValueError("Invalid ePC-SAFT fugacity coefficients")
            return value

        def molecular_gibbs(n, species, mu0, phase):
            if np.sum(n)<=1e-13: return 0.
            positive=np.maximum(n,1e-14)
            model,_=build_pcsaft_model(species)
            phis=np.asarray(cl.fugacity_coefficient(model,P,T,positive,phase=phase),dtype=float).reshape(-1)
            if np.any(~np.isfinite(phis)) or np.any(phis<=0): raise ValueError("Invalid PC-SAFT fugacities")
            mu=mu0+R*T*np.log(positive/positive.sum()*P/P0*phis)
            return float(np.dot(n,mu))

        def aqueous_gibbs(n):
            if n[0] <= 1e-9:
                # The water-based molal standard state requires solvent water.
                if np.sum(n[1:])>1e-10: raise ValueError("Aqueous solutes without water")
                return 0.
            phi=phi_electrolyte(n)
            q=np.maximum(n,1e-14); x=q/q.sum(); xw=x[0]
            mu_neutral=mu0_aq_neutral+R*T*np.log(x[:len(AQ_NEUTRAL_SPECIES)]*P/P0*phi[:len(AQ_NEUTRAL_SPECIES)])
            molal=np.maximum(n[-len(AQ_ION_SPECIES):],1e-14)/(MW_WATER_KG*n[0])
            mu_ions=mu0_ion+R*T*(np.log(molal)+np.log(xw)+np.log(phi[-len(AQ_ION_SPECIES):])-lnphi_infinity[-len(AQ_ION_SPECIES):])
            return float(np.dot(n[:len(AQ_NEUTRAL_SPECIES)],mu_neutral)+np.dot(n[-len(AQ_ION_SPECIES):],mu_ions))

        def phase_gibbs(n):
            n=np.asarray(n,dtype=float)
            gas=n[PHASE_INDEX["gas"]]; aq=n[PHASE_INDEX["aqueous"]]
            org=n[PHASE_INDEX["organic liquid"]]; solid=n[PHASE_INDEX["solid"]]
            return {"gas":molecular_gibbs(gas,GAS_SPECIES,mu0_gas,"vapour"),
                    "aqueous":aqueous_gibbs(aq),
                    "organic liquid":molecular_gibbs(org,ORG_SPECIES,mu0_org,"liquid"),
                    "solid":float(np.dot(solid,mu0_solid))}

        def total_gibbs(n):
            if np.any(np.asarray(n)<-1e-11): raise ValueError("Negative mole number")
            values=phase_gibbs(np.maximum(n,0.))
            if not np.all(np.isfinite(list(values.values()))): raise ValueError("Nonfinite phase Gibbs energy")
            return float(sum(values.values()))

        # ============================================================
        # FOUR-PHASE CELLULOSE EXAMPLE
        #
        # Feed per 1 mol cellulose carbon:
        #   1/6 mol cellulose
        #   10 mol water
        #   0.05 mol NaHCO3
        #
        # Carbon allocation is prescribed for this demonstration.
        # Species amounts within each phase are optimized.
        # ============================================================

        import numpy as np
        import pandas as pd
        from scipy.optimize import linprog, minimize
        from IPython.display import display

        if not (
            np.isclose(T, 623.15)
            and np.isclose(P, 25.0e6)
        ):
            raise ValueError(
                "Run the model-definition cells at 350 °C and "
                "250 bar before this example. Ion reference states "
                "and the dielectric model must match T and P."
            )

        # ------------------------------------------------------------
        # 1. Feed inventory
        # ------------------------------------------------------------

        feed = {
            ("solid", "cellulose"): 1.0 / 6.0,
            ("aqueous", "water"): 10.0,
            ("aqueous", "Na+"): 0.05,
            ("aqueous", "HCO3-"): 0.05,
        }

        feed_vector = np.zeros(len(PHASES))

        for key, amount in feed.items():
            feed_vector[PHASES.index(key)] = amount

        # A_original contains the elemental rows and charge row
        # constructed by the notebook.
        feed_balances = A_original @ feed_vector

        feed_balance_table = pd.DataFrame({
            "Balance": ELEMENTS + ["charge"],
            "Feed amount [mol]": feed_balances,
        })

        print("Feed: 1 mol cellulose C + 10 mol water + 0.05 mol NaHCO3")
        display(feed_balance_table)

        feed_carbon = feed_balances[ELEMENTS.index("C")]

        # ------------------------------------------------------------
        # 2. Demonstration targets
        #
        # These are prescribed carbon amounts, not predictions
        # of unconstrained phase stability.
        # ------------------------------------------------------------

        phase_carbon_fractions = {
            "gas": 0.55,
            "aqueous": 0.20,
            "organic liquid": 0.10,
            "solid": 0.15,
        }

        if not np.isclose(sum(phase_carbon_fractions.values()), 1.0):
            raise ValueError("Phase carbon fractions must sum to one.")

        carbon_atoms = A_original[ELEMENTS.index("C")]

        constraint_rows = [A_original]
        constraint_targets = [feed_balances]

        for phase in phase_carbon_fractions:
            mask = np.array(
                [p == phase for p, _ in PHASES],
                dtype=float,
            )
            constraint_rows.append(
                (carbon_atoms * mask)[None, :]
            )
            constraint_targets.append(
                np.array([
                    phase_carbon_fractions[phase] * feed_carbon
                ])
            )

        A_example = np.vstack(constraint_rows)
        b_example = np.concatenate(constraint_targets)

        # Require water-rich aqueous solution and visible gas.
        # The carbon targets already require all four phases.
        minimum = np.zeros(len(PHASES))
        minimum[PHASES.index(("aqueous", "water"))] = 8.0
        minimum[PHASES.index(("gas", "water"))] = 0.05
        minimum[PHASES.index(("aqueous", "Na+"))] = 0.05

        bounds_example = [
            (float(value), None)
            for value in minimum
        ]

        # ------------------------------------------------------------
        # 3. Obtain a feasible four-phase starting composition
        # ------------------------------------------------------------

        feasible = linprog(
            np.zeros(len(PHASES)),
            A_eq=A_example,
            b_eq=b_example,
            bounds=bounds_example,
            method="highs",
        )

        if not feasible.success:
            raise ValueError(
                "The proposed four-phase allocation is infeasible: "
                + feasible.message
            )

        print(
            "Maximum starting balance residual:",
            np.max(np.abs(A_example @ feasible.x - b_example)),
        )

        # ------------------------------------------------------------
        # 4. Minimize Gibbs energy subject to the demonstration
        #    carbon allocation
        # ------------------------------------------------------------

        G_start = total_gibbs(feasible.x)

        def example_objective(n):
            try:
                # Subtract the baseline to resolve smaller changes in G.
                return (total_gibbs(n) - G_start) / 1e4
            except ValueError as exc:
                if not any(
                    phrase in str(exc)
                    for phrase in (
                        "Invalid PC-SAFT fugacities",
                        "Invalid ePC-SAFT fugacity coefficients",
                    )
                ):
                    raise
                return 1e6 + np.sum((n - feasible.x) ** 2)

        result_example = minimize(
            example_objective,
            feasible.x,
            method="SLSQP",
            bounds=bounds_example,
            constraints=[{
                "type": "eq",
                "fun": lambda n: A_example @ n - b_example,
                "jac": lambda n: A_example,
            }],
            options={
                "maxiter": 1200,
                "ftol": 1e-10,
                "eps": 1e-7,
                "disp": False,
            },
        )

        balance_error = np.max(
            np.abs(A_example @ result_example.x - b_example)
        )

        print("Solver:", result_example.message)
        print("Maximum balance residual:", balance_error)

        if (
            not result_example.success
            or balance_error > 1e-7
            or np.min(result_example.x) < -1e-9
        ):
            raise RuntimeError(
                "The four-phase example did not produce a "
                "validated optimized result."
            )

        n_example = np.maximum(result_example.x, 0.0)

        # Verify with a direct EOS calculation, without the penalty.
        G_example = total_gibbs(n_example)
        phase_energies_example = phase_gibbs(n_example)

        # ------------------------------------------------------------
        # 5. Display every species and every phase
        # ------------------------------------------------------------

        species_table = pd.DataFrame({
            "Phase": [phase for phase, _ in PHASES],
            "Species": [species for _, species in PHASES],
            "Amount [mol/mol cellulose C]": n_example,
        })

        phase_rows = []

        for phase, fraction in phase_carbon_fractions.items():
            indices = PHASE_INDEX[phase]

            phase_rows.append({
                "Phase": phase,
                "Total amount [mol/mol cellulose C]":
                    float(np.sum(n_example[indices])),
                "Carbon [mol/mol cellulose C]":
                    float(carbon_atoms[indices] @ n_example[indices]),
                "Prescribed fraction of feed carbon": fraction,
                "Gibbs energy [J/mol cellulose C]":
                    phase_energies_example[phase],
            })

        print("\nPhase results:")
        display(pd.DataFrame(phase_rows))

        print("\nSpecies results:")
        display(species_table)

        print("\nTotal Gibbs energy [J/mol cellulose C]:", G_example)
        return {
            'species_table': species_table,
            'phase_table': pd.DataFrame(phase_rows),
            'solver_result': result_example,
            'amounts': n_example,
            'gibbs': G_example,
            'balance_residual': balance_error,
        }
    finally:
        model.__dict__.clear()
        model.__dict__.update(saved_model_state)


four_phase_original_result = None
four_phase_original_result = run_original_four_phase_example()


In [ ]:
if four_phase_original_result is None:
    raise RuntimeError('Run the original four-phase calculation successfully first.')

# ============================================================
# FOUR-PHASE EXAMPLE — SPECIES AMOUNTS BY PHASE
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

phase_order = ("gas", "aqueous", "organic liquid", "solid")
phase_colors = {
    "gas": "#4C78A8",
    "aqueous": "#54A24B",
    "organic liquid": "#F58518",
    "solid": "#B279A2",
}

fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 10),
    layout="constrained",
)

for ax, phase in zip(axes.flat, phase_order):
    data = four_phase_original_result["species_table"].loc[
        four_phase_original_result["species_table"]["Phase"] == phase
    ]

    names = data["Species"].to_list()
    amounts = data[
        "Amount [mol/mol cellulose C]"
    ].to_numpy(dtype=float)

    # Values below this threshold are treated as absent
    # in the logarithmic plot.
    visible = amounts >= 1e-10

    ax.bar(
        np.arange(len(names))[visible],
        amounts[visible],
        color=phase_colors[phase],
    )

    ax.set_xticks(np.arange(len(names)), names)
    ax.tick_params(axis="x", rotation=45)
    ax.set_yscale("log")
    ax.set_ylabel("Amount [mol/mol cellulose C]")
    ax.set_title(phase)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")

fig.suptitle(
    "Four-phase cellulose example: species amounts",
    fontsize=15,
)
plt.show()

## 4. full_text: ions, database minerals, and the 390 °C aqueous cut-off
Keep the Figures 7–8 feed, 240 bar, the per-temperature initial water choices, and exactly 80% gas carbon. Below 390 °C ions may occupy the aqueous phase and compete with selected minerals. At/above 390 °C aqueous is removed and all feed elements must be conserved in neutral molecular or solid products. The gas label also represents the retained dense/supercritical fluid calculation above the imposed cut-off; this is not a predicted mixture critical boundary.

Artinite and Hydromagnesite are hydrate records in your XML. Brucite is a hydroxide; Magnesite, Periclase, Halite, Sylvite and the alkali oxides are competing solids. Exact names and coefficients are loaded from your XML. Li/Cs have no mineral records in this XML: nonzero LiCl or CsCl additions require additional supported neutral products. MgCl2 also requires a compatible chloride destination; feasibility depends on the complete feed inventory. Salt additions remain zero by default, as in your uploaded case. No ions have been deleted from the feed or converted to unbalanced products.

Inspect the database coverage and numerical diagnostics before interpreting results. Carbon-containing minerals share the carbon balance with gas, organics and char; `solid_carbon_target="CHAR"` continues to refer to char alone.


In [ ]:
# ============================================================
# full_text — ALL CASE INPUTS (EDIT HERE)
# ============================================================
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
from IPython.display import display

FULL_TEXT_SETTINGS = {
    # Figures 7–8 temperature grid, in Celsius; pressure is absolute bar.
    'temperatures_C': [100, 150, 200, 250, 300, 340, 360, 380, 389.6,
                       390, 400, 420, 460, 500, 540, 580],
    'pressure_bar': 240.0,
    'aqueous_cutoff_C': 380.0,  # Same imposed cut-off as Figures 7–8.
    'aqueous_enabled': True,  # Set False to disable at every temperature.
    'ion_product_species': {
        'gas': [],  # Add only complete neutral NASA/SAFT names, if needed.
        'solid': ['Artinite', 'Hydromagnesite', 'Brucite', 'Magnesite',
                  'Periclase', 'Halite', 'Sylvite', 'Sodium-Oxide',
                  'Potassium-Oxide'],
    },

    # Reduced dry-manure elemental mass percentages (original Figure 7–8
    # atom inventory converted on 25000 g dry feed). Their sum is <100
    # because omitted elements are outside this reduced feed model.
    'feed_element_wt_percent': {
        'C': 30.3950366, 'H': 4.40810496, 'O': 53.57873112,
        'N': 2.66469168, 'S': 0.506548},
    'feed_species_moles': None,  # Alternative dry molar mixture; overrides wt%.
    'water_g_per_g_dry_feed': 3.999935304,  # Added water, separate from dry feed.
    'atomic_weights': {
        'C': 12.011, 'H': 1.008, 'O': 15.999, 'N': 14.007, 'S': 32.06,
        'Na': 22.989769, 'K': 39.0983, 'Cs': 132.90545196,
        'Li': 6.94, 'Mg': 24.305, 'Cl': 35.45},

    # Salt additions in mol/g DRY feed, not mol/kg and not mol/mol C.
    # Defaults of zero preserve the reduced Figure 7–8 inventory.
    # For example, 1e-5 mol/g = 0.01 mol/kg dry feed.
    # Salts supply ELEMENTS; their molecular identity is not constrained
    # to remain in the equilibrium products.
    'added_salts': {
        'NaHCO3': {'mol_per_g_dry_feed': 0.0,
                   'elements': {'Na': 1, 'H': 1, 'C': 1, 'O': 3}},
        'KCl': {'mol_per_g_dry_feed': 0.0, 'elements': {'K': 1, 'Cl': 1}},
        'CsCl': {'mol_per_g_dry_feed': 0.0, 'elements': {'Cs': 1, 'Cl': 1}},
        'LiCl': {'mol_per_g_dry_feed': 0.0, 'elements': {'Li': 1, 'Cl': 1}},
        'MgCl2': {'mol_per_g_dry_feed': 0.0, 'elements': {'Mg': 1, 'Cl': 2}},
    },

    # Allowed PRODUCTS in each phase. Species may be edited using supported
    # database names in `available`. Ions require aqueous water and the
    # electrolyte model. A feed element needs at least one product carrier.
    'phase_species': {
        'gas': ['water', 'hydrogen', 'carbon dioxide', 'carbon monoxide',
                'methane', 'methanol', 'ammonia', 'hydrogen sulfide', 'nitrogen'],
        'aqueous': ['water', 'hydrogen', 'carbon dioxide', 'carbon monoxide',
                    'methane', 'methanol', 'ammonia', 'hydrogen sulfide',
                    'Na+', 'HCO3-', 'CO3^2-', 'K+', 'Cs+', 'Cl-', 'Li+', 'Mg2+'],
        'organic liquid': ['benzene', 'toluene'],
        'solid': ['carbon'],  # Mineral candidates above are merged by the model.
    },

    # Equality: exactly 80% of TOTAL feed C enters gas products.
    # Includes any carbon supplied in added salts. Set None for free allocation.
    'gas_carbon_fraction': 0.8,
    'phase_carbon_fractions': {},  # No separate aqueous/organic/solid targets.
    'solid_carbon_fraction': None,
    'solid_carbon_target': 'CHAR',

    'aqueous_model': 'electrolyte',
    # Numerical floor in mol water/mol DRY-feed carbon; not a concentration limit.
    'aqueous_water_floor': 1e-6,
    'minimum_amounts': {},  # Optional permanent {(phase, species): amount} bounds.

    # Per-temperature starting choices below override these two flags.
    'liquid_initialization': True,
    'gas_water_initialization': False,
    'initial_gas_water_fraction': 1e-6,  # Initial liquid-start cap only.
    'initial_phase_amounts': None,      # No manually prescribed starting amounts.

    # Same default numerical controls as the Figure 7–8 shared solver.
    # ftol: scaled objective convergence tolerance; eps: absolute mole step.
    'optimizer_ftol': 1e-9,
    'optimizer_eps': 1e-7,
    'max_iterations': 5000,
    # No optimizer_energy_scale override: shared solver uses
    # max(10000, abs(initial Gibbs)), in J/mol dry-feed carbon.
}

# Water starting side for EACH temperature. These choices do not remove phases.
# 'liquid': maximize initial aqueous water and cap initial gas water.
# 'vapor': maximize initial gas water while respecting aqueous solvent floor.
FULL_TEXT_INITIALIZATION = {
    100.0: 'liquid', 150.0: 'liquid', 200.0: 'liquid', 250.0: 'liquid',
    300.0: 'liquid', 340.0: 'liquid', 360.0: 'liquid', 380.0: 'liquid',
    389.6: 'liquid', 390.0: 'vapor', 400.0: 'vapor', 420.0: 'vapor',
    460.0: 'vapor', 500.0: 'vapor', 540.0: 'vapor', 580.0: 'vapor',
}
print('full_text: complete case definition')
pprint(FULL_TEXT_SETTINGS, sort_dicts=False, width=100)
print('Initial water side by temperature:')
pprint(FULL_TEXT_INITIALIZATION, sort_dicts=False)
print('Fluid-water ideal reference: NASA H2O in HTG_Model.')

print('Mineral data from your XML (hydrate identity and equation limits):')
display(model.mineral_inventory(elements=['Na', 'K', 'Li', 'Cs', 'Mg']))
print('Charged ions remain aqueous-only; neutral product coverage at 400 C:')
display(model.ion_product_candidates(elements=['Na', 'K', 'Li', 'Cs', 'Cl', 'Mg'], temperature_C=400))


In [ ]:
# ============================================================
# FULL_TEXT SWEEP — BALANCES AND NUMERICAL DIAGNOSTICS
# Run the case-definition cell again after editing inputs.
# ============================================================
full_text_results = []
full_text_diagnostics = []
full_text_species_rows = []
full_text_ion_rows = []
saved_AW = dict(model.AW)
try:
    model.AW = {**model.AW, **FULL_TEXT_SETTINGS['atomic_weights']}
    for temperature in FULL_TEXT_SETTINGS['temperatures_C']:
        settings = copy.deepcopy(FULL_TEXT_SETTINGS)
        settings['temperature_C'] = float(temperature)
        mode = FULL_TEXT_INITIALIZATION.get(float(temperature))
        if mode not in ('liquid', 'vapor'):
            raise ValueError(f'{temperature:g} C: choose liquid or vapor initialization')
        settings['liquid_initialization'] = mode == 'liquid'
        settings['gas_water_initialization'] = mode == 'vapor'
        settings, allocation = model.prepare_phase_allocation(settings)
        if allocation['aqueous_disabled']:
            mode = 'vapor (aqueous disabled)'
            settings['gas_water_initialization'] = True
        print(f"\nT = {temperature:g} °C; P = {settings['pressure_bar']:g} bar")
        for phase, names in settings['phase_species'].items():
            print(f'{phase}: {names}')
        diagnostic = {'T_C': temperature, 'P_bar': settings['pressure_bar'], 'Initialization': mode,
                      'Aqueous_disabled': allocation['aqueous_disabled']}
        try:
            result = model.solve_equilibrium(settings)
            full_text_results.append(result)
            water_amount = result['amounts'].get(('aqueous', 'water'), 0.0)
            water_kg_per_C = water_amount * 0.01801528  # Same mass convention as aqueous_gibbs.
            for ion in settings['phase_species']['aqueous']:
                if ion in model.ION_DATA:
                    full_text_ion_rows.append({'T_C': temperature, 'Ion': ion,
                        'Amount_mol_per_dry_C': result['amounts'].get(('aqueous', ion), 0.0),
                        'Molality_mol_per_kg_water': result['amounts'].get(('aqueous', ion), 0.0) / water_kg_per_C
                            if water_kg_per_C > 0 else np.nan})
            diagnostic.update(Status='solved', Error='',
                Balance_residual=result['balance_residual'], Charge_residual=result['charge_residual'], Iterations=result['iterations'],
                G_J_per_mol_dry_C=result['gibbs_J_per_mol_feed_C'],
                Recorded_EOS_errors=len(result['eos_trial_errors']),
                EOS_messages='; '.join(result['eos_trial_errors']))
            for (phase, species), amount in result['amounts'].items():
                full_text_species_rows.append({'T_C': temperature, 'Phase': phase,
                                             'Species': species, 'Amount_mol_per_dry_C': amount})
        except Exception as exc:
            diagnostic.update(Status='failed', Error=str(exc), Balance_residual=np.nan,
                Iterations=np.nan, G_J_per_mol_dry_C=np.nan, Recorded_EOS_errors=np.nan,
                EOS_messages='')
        full_text_diagnostics.append(diagnostic)
        print(diagnostic['Status'])
finally:
    model.AW = saved_AW
full_text_diagnostics = pd.DataFrame(full_text_diagnostics)
full_text_species_results = pd.DataFrame(full_text_species_rows,
    columns=['T_C', 'Phase', 'Species', 'Amount_mol_per_dry_C'])
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(full_text_diagnostics)
    display(full_text_species_results)
# `Recorded_EOS_errors` counts stored messages (capped at 10), not all trials.
# Failed points remain undefined in the following plots.

full_text_ion_molalities = pd.DataFrame(full_text_ion_rows, columns=['T_C', 'Ion', 'Amount_mol_per_dry_C', 'Molality_mol_per_kg_water'])
print('Ion molalities (inspect before accepting solvent-floor choices):')
with pd.option_context('display.max_rows', None):
    display(full_text_ion_molalities)


In [ ]:
# ============================================================
# FULL_TEXT SWEEP — PHASE DISTRIBUTION AND SPECIES COMPOSITION
# ============================================================
full_text_phases = ['gas', 'aqueous', 'organic liquid', 'solid']
full_text_T = pd.Index(FULL_TEXT_SETTINGS['temperatures_C'], name='T_C')
full_text_phase_amounts = pd.DataFrame(np.nan, index=full_text_T, columns=full_text_phases)
full_text_phase_carbon = full_text_phase_amounts.copy()
full_text_compositions = {
    phase: pd.DataFrame(np.nan, index=full_text_T,
                         columns=FULL_TEXT_SETTINGS['phase_species'][phase])
    for phase in full_text_phases}
for result in full_text_results:
    t = result['temperature_C']
    total_feed_C = result['feed_atoms'][result['elements'].index('C')]
    for phase in full_text_phases:
        names = FULL_TEXT_SETTINGS['phase_species'][phase]
        amounts = np.array([result['amounts'].get((phase, name), 0.0) for name in names])
        registry = model.SOLID_DATA if phase == 'solid' else {
            **model.MOLECULAR_SPECIES, **model.ORGANIC_SPECIES, **model.ION_DATA}
        full_text_phase_amounts.loc[t, phase] = amounts.sum()
        full_text_phase_carbon.loc[t, phase] = sum(
            amount * registry[name]['elements'].get('C', 0.0)
            for name, amount in zip(names, amounts)) / total_feed_C
        if amounts.sum() > 1e-12:
            full_text_compositions[phase].loc[t] = amounts / amounts.sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
for ax, frame, title, ylabel in [
    (axes[0], full_text_phase_amounts, 'Phase amounts', 'mol/mol dry-feed C'),
    (axes[1], full_text_phase_carbon, 'Carbon allocation', 'Fraction of total feed carbon')]:
    for phase in full_text_phases:
        ax.plot(frame.index, frame[phase], 'o-', label=phase)
    ax.set(title=title, xlabel='Temperature [°C]', ylabel=ylabel,
           xlim=(full_text_T.min(), full_text_T.max()))
    ax.grid(alpha=.25); ax.legend()
axes[1].set_ylim(0, 1.05)
fig.suptitle(f'full_text: reduced Figures 7–8 feed + selected ions — {FULL_TEXT_SETTINGS["pressure_bar"]:g} bar')
plt.show()
fig, axes = plt.subplots(2, 2, figsize=(15, 10), layout='constrained')
for ax, phase in zip(axes.flat, full_text_phases):
    for name in full_text_compositions[phase].columns:
        ax.plot(full_text_T, full_text_compositions[phase][name], 'o-', ms=3, label=name)
    ax.set(title=phase, xlabel='Temperature [°C]', ylabel='Species mole fraction in phase',
           ylim=(0, 1.05), xlim=(full_text_T.min(), full_text_T.max()))
    ax.grid(alpha=.25)
    if len(full_text_compositions[phase].columns): ax.legend(fontsize=7)
    if not full_text_compositions[phase].notna().any().any():
        ax.text(.5, .5, 'Phase absent or no solved states', ha='center', va='center',
                transform=ax.transAxes)
fig.suptitle('Phase compositions including water; solids use formula-unit fractions')
plt.show()


In [ ]:
# Element allocation to phases, including hydrate water and salt elements.
# Every row uses mol atoms / mol dry-feed carbon. Feed vs product is checked.
full_text_element_rows = []
for result in full_text_results:
    registry = {**model.ALL_MOLECULAR_DATA, **model.SOLID_DATA, **model.ION_DATA}
    for element, feed in zip(result['elements'], result['feed_atoms']):
        if feed <= 0:
            continue
        row = {'T_C':result['temperature_C'], 'Element':element, 'Feed_atoms':feed}
        for phase in ['gas', 'aqueous', 'organic liquid', 'solid']:
            row[phase] = sum(amount*registry[name]['elements'].get(element, 0)
                for (p,name),amount in result['amounts'].items() if p == phase)
        row['Residual'] = sum(row[p] for p in ['gas','aqueous','organic liquid','solid'])-feed
        full_text_element_rows.append(row)
full_text_element_allocation = pd.DataFrame(full_text_element_rows)
display(full_text_element_allocation)


In [ ]:
# CARBON DISTRIBUTION BETWEEN PHASES
import numpy as np
import pandas as pd
from IPython.display import display

registry = {
    **model.ALL_MOLECULAR_DATA,
    **model.SOLID_DATA,
    **model.ION_DATA,
}
phases = ["gas", "aqueous", "organic liquid", "solid"]
rows = []

for result in full_text_results:
    feed_carbon = float(
        result["feed_atoms"][result["elements"].index("C")]
    )

    # Carbon atoms in each phase [mol C / mol dry-feed C].
    carbon = {phase: 0.0 for phase in phases}
    for (phase, species), amount in result["amounts"].items():
        carbon[phase] += (
            amount * registry[species]["elements"].get("C", 0.0)
        )

    product_carbon = sum(carbon.values())

    rows.append({
        "T [°C]": result["temperature_C"],
        "Aqueous allowed": bool(
            result["phase_allocation"]["effective_phase_species"]
            .get("aqueous", [])
        ),
        **{
            f"{phase} C [%]": 100 * carbon[phase] / feed_carbon
            for phase in phases
        },
        "Total C [%]": 100 * product_carbon / feed_carbon,
        "C balance residual [mol/mol dry C]":
            product_carbon - feed_carbon,
    })

carbon_distribution = pd.DataFrame(rows)

if carbon_distribution.empty:
    print("No successful full_text results available.")
else:
    carbon_distribution = carbon_distribution.sort_values("T [°C]")

    display(carbon_distribution.style.format({
        "T [°C]": "{:.1f}",
        **{f"{phase} C [%]": "{:.4f}" for phase in phases},
        "Total C [%]": "{:.6f}",
        "C balance residual [mol/mol dry C]": "{:.3e}",
    }))

    print(
        "Carbon balance:",
        "PASS" if np.allclose(
            carbon_distribution["Total C [%]"],
            100.0, rtol=0, atol=1e-4
        ) else "CHECK RESIDUALS"
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

registry = {
    **model.ALL_MOLECULAR_DATA,
    **model.SOLID_DATA,
    **model.ION_DATA,
}
phases = ["gas", "aqueous", "organic liquid", "solid"]
rows = []

for result in full_text_results:
    feed_C = float(
        result["feed_atoms"][result["elements"].index("C")]
    )

    for (phase, species), amount in result["amounts"].items():
        n_carbon = registry[species]["elements"].get("C", 0.0)
        if n_carbon > 0:
            rows.append({
                "T_C": result["temperature_C"],
                "Phase": phase,
                "Species": species,
                "Feed_carbon_percent":
                    100.0 * amount * n_carbon / feed_C,
            })

carbon_species = pd.DataFrame(rows)

if carbon_species.empty:
    print("No successful carbon-containing results available.")
else:
    # Reindex to the requested grid so failed temperatures remain gaps.
    temperatures = sorted(set(FULL_TEXT_SETTINGS["temperatures_C"]))
    cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")

    fig, axes = plt.subplots(
        2, 2, figsize=(13, 9), sharex=True,
        layout="constrained"
    )

    for ax, phase in zip(axes.flat, phases):
        table = (
            carbon_species.loc[carbon_species["Phase"] == phase]
            .pivot_table(
                index="T_C", columns="Species",
                values="Feed_carbon_percent", aggfunc="sum"
            )
            .reindex(temperatures)
        )

        # Explicitly show zero aqueous carbon where aqueous was removed.
        if phase == "aqueous":
            for result in full_text_results:
                if not result["phase_allocation"][
                    "effective_phase_species"
                ].get("aqueous", []):
                    table.loc[result["temperature_C"], :] = 0.0

        for species in table.columns:
            values = table[species]
            if values.max(skipna=True) > 1e-6:
                ax.plot(
                    table.index, values,
                    marker="o", markersize=4, label=species
                )

        if cutoff is not None:
            ax.axvline(
                cutoff, color="black", linestyle="--",
                linewidth=1, alpha=0.6
            )

        ax.set_title(phase.capitalize())
        ax.set_ylabel("Carbon [% of total feed C]")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)

        if ax.lines and ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
        else:
            ax.text(
                0.5, 0.5, "No carbon species above plotting threshold",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9
            )

    for ax in axes[-1]:
        ax.set_xlabel("Temperature [°C]")

    fig.suptitle(
        "Carbon distribution by species and phase\n"
        "Dashed line: aqueous cut-off; gaps: failed calculations"
    )
    plt.show()

In [ ]:
from pathlib import Path
import json

# Search the working directory and its upload subfolder.
candidates = []

for folder in [Path("."), Path("upload")]:
    if not folder.is_dir():
        continue

    for path in folder.glob("*.ipynb"):
        try:
            notebook = json.loads(path.read_text(encoding="utf-8"))

            # Check the solver implementation, not just the capability flag.
            supported = any(
                cell.get("cell_type") == "code"
                and "def solve_equilibrium(" in "".join(cell["source"])
                and "gas_carbon_min_fraction" in "".join(cell["source"])
                and "gas_min_row" in "".join(cell["source"])
                and "optimizer_constraints" in "".join(cell["source"])
                for cell in notebook["cells"]
            )

            if supported:
                candidates.append(path.resolve())

        except (OSError, ValueError, KeyError):
            continue

if not candidates:
    raise RuntimeError(
        f"No updated model found in {Path.cwd()} or its upload folder. "
        "Download the updated HTG_Model notebook from the previous message "
        "and place it beside the Cases notebook."
    )

MODEL_NOTEBOOK = max(candidates, key=lambda p: p.stat().st_mtime)
print("Loading model from:", MODEL_NOTEBOOK)

model = load_model_notebook(MODEL_NOTEBOOK)
model.initialize_model(
    DATABASE_PATHS,
    species_overrides=globals().get("SPECIES_OVERRIDES", {}),
)

assert getattr(model, "SUPPORTS_GAS_CARBON_MIN_FRACTION", False), (
    "The selected notebook did not load the capability flag."
)

print("Minimum gas-carbon constraint support loaded successfully.")

In [ ]:
# ============================================================
# full_text — ALL CASE INPUTS (EDIT HERE)
# ============================================================
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
from IPython.display import display

FULL_TEXT_SETTINGS = {
    # Figures 7–8 temperature grid, in Celsius; pressure is absolute bar.
    'temperatures_C': [100, 150, 200, 250, 300, 340, 360, 380, 389.6,
                       390, 400, 420, 460, 500, 540, 580],
    'pressure_bar': 240.0,
    'aqueous_cutoff_C': 390.0,  # Same imposed cut-off as Figures 7–8.
    'aqueous_enabled': True,  # Set False to disable at every temperature.
    'ion_product_species': {'gas': [], 'solid': []},  # Populated below from your XML.

    # Reduced dry-manure elemental mass percentages (original Figure 7–8
    # atom inventory converted on 25000 g dry feed). Their sum is <100
    # because omitted elements are outside this reduced feed model.
    'feed_element_wt_percent': {
        'C': 30.3950366, 'H': 4.40810496, 'O': 53.57873112,
        'N': 2.66469168, 'S': 0.506548},
    'feed_species_moles': None,  # Alternative dry molar mixture; overrides wt%.
    'water_g_per_g_dry_feed': 3.999935304,  # Added water, separate from dry feed.
    'atomic_weights': {
        'C': 12.011, 'H': 1.008, 'O': 15.999, 'N': 14.007, 'S': 32.06,
        'Na': 22.989769, 'K': 39.0983, 'Cs': 132.90545196,
        'Li': 6.94, 'Mg': 24.305, 'Cl': 35.45},

    # Salt additions in mol/g DRY feed, not mol/kg and not mol/mol C.
    # Small EXAMPLE additions exercise Na/K/Mg/Cl partitioning.
    # Set NaHCO3, KCl and MgCO3 back to zero for the original salt-free feed.
    # For example, 1e-5 mol/g = 0.01 mol/kg dry feed.
    # Salts supply ELEMENTS; their molecular identity is not constrained
    # to remain in the equilibrium products.
    'added_salts': {
        'NaHCO3': {'mol_per_g_dry_feed': 1e-5,
                   'elements': {'Na': 1, 'H': 1, 'C': 1, 'O': 3}},
        'KCl': {'mol_per_g_dry_feed': 1e-5, 'elements': {'K': 1, 'Cl': 1}},
        'CsCl': {'mol_per_g_dry_feed': 0.0, 'elements': {'Cs': 1, 'Cl': 1}},
        'LiCl': {'mol_per_g_dry_feed': 0.0, 'elements': {'Li': 1, 'Cl': 1}},
        'MgCl2': {'mol_per_g_dry_feed': 0.0, 'elements': {'Mg': 1, 'Cl': 2}},
        # MgCO3 introduces Mg without excess chloride that lacks a carrier.
        'MgCO3': {'mol_per_g_dry_feed': 1e-5, 'elements': {'Mg': 1, 'C': 1, 'O': 3}},
    },

    # Allowed PRODUCTS in each phase. Species may be edited using supported
    # database names in `available`. Ions require aqueous water and the
    # electrolyte model. A feed element needs at least one product carrier.
    'phase_species': {
        'gas': ['water', 'hydrogen', 'carbon dioxide', 'carbon monoxide',
                'methane', 'methanol', 'ammonia', 'hydrogen sulfide', 'nitrogen'],
        'aqueous': ['water', 'hydrogen', 'carbon dioxide', 'carbon monoxide',
                    'methane', 'methanol', 'ammonia', 'hydrogen sulfide',
                    ],  # Complete feed-compatible ions are appended below.
        'organic liquid': ['benzene', 'toluene'],
        'solid': ['carbon'],  # Mineral candidates above are merged by the model.
    },

    # Equality: exactly 90% of TOTAL feed C enters gas products.
    # Includes any carbon supplied in added salts. Set None for free allocation.
    'gas_carbon_fraction': 0.9,
    'phase_carbon_fractions': {},  # No separate aqueous/organic/solid targets.
    'solid_carbon_fraction': None,
    'solid_carbon_target': 'CHAR',

    'aqueous_model': 'electrolyte',
    # Numerical floor in mol water/mol DRY-feed carbon; not a concentration limit.
    'aqueous_water_floor': 1e-6,
    'minimum_amounts': {},  # Optional permanent {(phase, species): amount} bounds.

    # Per-temperature starting choices below override these two flags.
    'liquid_initialization': True,
    'gas_water_initialization': False,
    'initial_gas_water_fraction': 1e-6,  # Initial liquid-start cap only.
    'initial_phase_amounts': None,      # No manually prescribed starting amounts.

    # Same default numerical controls as the Figure 7–8 shared solver.
    # ftol: scaled objective convergence tolerance; eps: absolute mole step.
    'optimizer_ftol': 1e-9,
    'optimizer_eps': 1e-7,
    'max_iterations': 5000,
    # No optimizer_energy_scale override: shared solver uses
    # max(10000, abs(initial Gibbs)), in J/mol dry-feed carbon.
}

# Water starting side for EACH temperature. These choices do not remove phases.
# 'liquid': maximize initial aqueous water and cap initial gas water.
# 'vapor': maximize initial gas water while respecting aqueous solvent floor.
FULL_TEXT_INITIALIZATION = {
    100.0: 'liquid', 150.0: 'liquid', 200.0: 'liquid', 250.0: 'liquid',
    300.0: 'liquid', 340.0: 'liquid', 360.0: 'liquid', 380.0: 'liquid',
    389.6: 'liquid', 390.0: 'vapor', 400.0: 'vapor', 420.0: 'vapor',
    460.0: 'vapor', 500.0: 'vapor', 540.0: 'vapor', 580.0: 'vapor',
}
# Rebuild the selection after changing feed, salt additions or temperatures.
# Only species with complete adapters and atoms actually supplied are selected.
def configure_full_text_species(case):
    configured = copy.deepcopy(case)
    feed = (model.feed_from_species(configured['feed_species_moles'])
            if configured.get('feed_species_moles') is not None
            else configured['feed_element_wt_percent'])
    atoms = {e: wt / 100.0 / model.AW[e] for e, wt in feed.items() if wt > 0}
    water = configured['water_g_per_g_dry_feed'] / (2*model.AW['H']+model.AW['O'])
    atoms['H'] = atoms.get('H', 0) + 2*water
    atoms['O'] = atoms.get('O', 0) + water
    for salt in configured['added_salts'].values():
        for e, count in salt['elements'].items():
            atoms[e] = atoms.get(e, 0) + salt['mol_per_g_dry_feed']*count
    active = {e for e, count in atoms.items() if count > 0}
    metals = active - {'C','H','O','N','S','Cl','F','Br','I'}
    temperatures_K = np.asarray(configured['temperatures_C']) + 273.15
    report, ions, solids = [], [], []
    catalogue = model.available_species(include_unavailable=True)
    ready_ions = set(catalogue.loc[
        catalogue.Phase.eq('aqueous ion') & catalogue.Ready, 'Species'])
    for name, data in model.ION_DATA.items():
        missing = set(data['elements']) - active
        reason = ('Elements not supplied: ' + ', '.join(sorted(missing)) if missing
                  else 'Incomplete HKF/electrolyte data' if name not in ready_ions else '')
        if not reason:
            ions.append(name)
        report.append({'Species':name, 'Kind':'aqueous ion',
                       'Selected':not bool(reason), 'Reason':reason or 'Complete HKF/EOS; feed-compatible'})
    for name, data in model.SOLID_DATA.items():
        if data.get('adapter') != 'supcrt_mineral':
            continue
        elements = set(data['elements'])
        if not data['hydrate'] and not elements.intersection(metals):
            continue
        missing = elements-active
        reason = ('Elements not supplied: ' + ', '.join(sorted(missing)) if missing
                  else 'Outside mineral equation range on the requested grid'
                  if temperatures_K.min() < 298.15 or temperatures_K.max() > data['supcrt_mineral']['Tmax'] else '')
        if not reason:
            solids.append(name)
        report.append({'Species':name, 'Kind':'hydrate' if data['hydrate'] else 'competing mineral',
                       'Selected':not bool(reason), 'Reason':reason or 'XML data valid over entire temperature grid'})
    # Show unsupported hydrate records as exclusions (e.g. Nesquehonite).
    for node in model.supcrt_root.iter('Species'):
        formula = node.findtext('Formula') or ''
        name = node.findtext('Name')
        if node.findtext('Type') == 'Mineral' and '*' in formula and 'H2O' in formula and name not in model.SOLID_DATA:
            record = catalogue[catalogue.Species.eq(name) & catalogue.Phase.eq('solid')]
            report.append({'Species':name, 'Kind':'hydrate', 'Selected':False,
                           'Reason':record.Reason.iloc[0] if not record.empty else 'No supported mineral adapter'})
    neutrals = [s for s in configured['phase_species']['aqueous'] if s not in model.ION_DATA]
    configured['phase_species']['aqueous'] = list(dict.fromkeys(neutrals + sorted(ions)))
    configured['ion_product_species']['solid'] = sorted(solids)
    return configured, pd.DataFrame(report), atoms


FULL_TEXT_SETTINGS, full_text_selection_report, full_text_feed_atoms_per_g = configure_full_text_species(FULL_TEXT_SETTINGS)
print('Selected ions, hydrates and competing solids; exclusions include the reason:')
display(full_text_selection_report)

# Check the COMPLETE nonnegative elemental/charge/carbon balances at EVERY
# temperature before invoking EOS or optimization. This does not test EOS validity.
full_text_preflight_rows = []
for temperature in FULL_TEXT_SETTINGS['temperatures_C']:
    candidate = copy.deepcopy(FULL_TEXT_SETTINGS)
    candidate['temperature_C'] = float(temperature)
    candidate, allocation = model.prepare_phase_allocation(candidate)
    pairs = [(phase, name) for phase, names in candidate['phase_species'].items() for name in names]
    registry = {**model.ALL_MOLECULAR_DATA, **model.SOLID_DATA, **model.ION_DATA}
    elements = sorted(set(full_text_feed_atoms_per_g) | {e for p,s in pairs for e in registry[s]['elements']})
    matrix = np.asarray([[registry[s]['elements'].get(e, 0) for p,s in pairs] for e in elements], dtype=float)
    targets = np.asarray([full_text_feed_atoms_per_g.get(e, 0) for e in elements])
    charge = np.asarray([model.ION_DATA[s]['charge'] if p == 'aqueous' and s in model.ION_DATA else 0 for p,s in pairs])
    carbon = matrix[elements.index('C')]
    gas_carbon = carbon * np.asarray([p == 'gas' for p,s in pairs])
    equality_rows, rhs_values = list(matrix) + [charge], list(targets) + [0.]
    if candidate.get('gas_carbon_fraction') is not None:
        equality_rows.append(gas_carbon)
        rhs_values.append(candidate['gas_carbon_fraction']*full_text_feed_atoms_per_g['C'])
    for phase, fraction in candidate.get('phase_carbon_fractions', {}).items():
        equality_rows.append(carbon*np.asarray([p == phase for p,s in pairs]))
        rhs_values.append(fraction*full_text_feed_atoms_per_g['C'])
    if candidate.get('solid_carbon_fraction') is not None:
        target = candidate.get('solid_carbon_target', 'CHAR')
        if target not in ('CHAR', 'all solids'):
            raise ValueError('Invalid solid_carbon_target')
        equality_rows.append(carbon*np.asarray([p == 'solid' and (target == 'all solids' or s == 'carbon') for p,s in pairs]))
        rhs_values.append(candidate['solid_carbon_fraction']*full_text_feed_atoms_per_g['C'])
    equality, rhs = np.asarray(equality_rows), np.asarray(rhs_values)
    feed_wt = (model.feed_from_species(candidate['feed_species_moles'])
               if candidate.get('feed_species_moles') is not None else candidate['feed_element_wt_percent'])
    dry_C_per_g = feed_wt['C']/100./model.AW['C']
    bounds = [(0., None)]*len(pairs)
    for pair, value in candidate.get('minimum_amounts', {}).items():
        bounds[pairs.index(pair)] = (value*dry_C_per_g, None)
    if ('aqueous','water') in pairs:
        j = pairs.index(('aqueous','water'))
        bounds[j] = (max(bounds[j][0], candidate['aqueous_water_floor']*dry_C_per_g), None)
    feasible = model.linprog(np.zeros(len(pairs)), A_eq=equality, b_eq=rhs, bounds=bounds, method='highs')
    full_text_preflight_rows.append({'T_C':temperature, 'Aqueous_disabled':allocation['aqueous_disabled'],
                                    'Balance_constraints_feasible':feasible.success, 'Message':feasible.message})
full_text_preflight = pd.DataFrame(full_text_preflight_rows)
display(full_text_preflight)
if not full_text_preflight.Balance_constraints_feasible.all():
    raise ValueError('full_text selection cannot conserve every feed element at every temperature. Inspect preflight and revise feed/product coverage; no feed element has been discarded.')


print('full_text: complete case definition')
pprint(FULL_TEXT_SETTINGS, sort_dicts=False, width=100)
print('Initial water side by temperature:')
pprint(FULL_TEXT_INITIALIZATION, sort_dicts=False)
print('Fluid-water ideal reference: NASA H2O in HTG_Model.')

print('Mineral data from your XML (hydrate identity and equation limits):')
display(model.mineral_inventory(elements=['Na', 'K', 'Li', 'Cs', 'Mg']))
print('Charged ions remain aqueous-only; neutral product coverage at 400 C:')
display(model.ion_product_candidates(elements=['Na', 'K', 'Li', 'Cs', 'Cl', 'Mg'], temperature_C=400))


In [ ]:
# ============================================================
# FULL_TEXT SWEEP — BALANCES AND NUMERICAL DIAGNOSTICS
# Run the case-definition cell again after editing inputs.
# ============================================================
full_text_results = []
full_text_diagnostics = []
full_text_species_rows = []
full_text_ion_rows = []
saved_AW = dict(model.AW)
try:
    model.AW = {**model.AW, **FULL_TEXT_SETTINGS['atomic_weights']}
    for temperature in FULL_TEXT_SETTINGS['temperatures_C']:
        settings = copy.deepcopy(FULL_TEXT_SETTINGS)
        settings['temperature_C'] = float(temperature)
        mode = FULL_TEXT_INITIALIZATION.get(float(temperature))
        if mode not in ('liquid', 'vapor'):
            raise ValueError(f'{temperature:g} C: choose liquid or vapor initialization')
        settings['liquid_initialization'] = mode == 'liquid'
        settings['gas_water_initialization'] = mode == 'vapor'
        settings, allocation = model.prepare_phase_allocation(settings)
        if allocation['aqueous_disabled']:
            mode = 'vapor (aqueous disabled)'
            settings['gas_water_initialization'] = True
        print(f"\nT = {temperature:g} °C; P = {settings['pressure_bar']:g} bar")
        for phase, names in settings['phase_species'].items():
            print(f'{phase}: {names}')
        diagnostic = {'T_C': temperature, 'P_bar': settings['pressure_bar'], 'Initialization': mode,
                      'Aqueous_disabled': allocation['aqueous_disabled']}
        try:
            result = model.solve_equilibrium(settings)
            full_text_results.append(result)
            water_amount = result['amounts'].get(('aqueous', 'water'), 0.0)
            water_kg_per_C = water_amount * 0.01801528  # Same mass convention as aqueous_gibbs.
            for ion in FULL_TEXT_SETTINGS['phase_species']['aqueous']:
                if ion in model.ION_DATA:
                    full_text_ion_rows.append({'T_C': temperature, 'Ion': ion,
                        'Amount_mol_per_dry_C': result['amounts'].get(('aqueous', ion), 0.0),
                        'Molality_mol_per_kg_water': result['amounts'].get(('aqueous', ion), 0.0) / water_kg_per_C
                            if water_kg_per_C > 0 else np.nan})
            diagnostic.update(Status='solved', Error='',
                Balance_residual=result['balance_residual'], Charge_residual=result['charge_residual'], Iterations=result['iterations'],
                G_J_per_mol_dry_C=result['gibbs_J_per_mol_feed_C'],
                Recorded_EOS_errors=len(result['eos_trial_errors']),
                EOS_messages='; '.join(result['eos_trial_errors']))
            for (phase, species), amount in result['amounts'].items():
                full_text_species_rows.append({'T_C': temperature, 'Phase': phase,
                                             'Species': species, 'Amount_mol_per_dry_C': amount})
        except Exception as exc:
            diagnostic.update(Status='failed', Error=str(exc), Balance_residual=np.nan,
                Iterations=np.nan, G_J_per_mol_dry_C=np.nan, Recorded_EOS_errors=np.nan,
                EOS_messages='')
        full_text_diagnostics.append(diagnostic)
        print(diagnostic['Status'])
finally:
    model.AW = saved_AW
full_text_diagnostics = pd.DataFrame(full_text_diagnostics)
full_text_species_results = pd.DataFrame(full_text_species_rows,
    columns=['T_C', 'Phase', 'Species', 'Amount_mol_per_dry_C'])
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(full_text_diagnostics)
    display(full_text_species_results)
# `Recorded_EOS_errors` counts stored messages (capped at 10), not all trials.
# Failed points remain undefined in the following plots.

full_text_ion_molalities = pd.DataFrame(full_text_ion_rows, columns=['T_C', 'Ion', 'Amount_mol_per_dry_C', 'Molality_mol_per_kg_water'])
print('Ion molalities (inspect before accepting solvent-floor choices):')
with pd.option_context('display.max_rows', None):
    display(full_text_ion_molalities)


In [ ]:
# ============================================================
# FULL_TEXT SWEEP — PHASE DISTRIBUTION AND SPECIES COMPOSITION
# ============================================================
full_text_phases = ['gas', 'aqueous', 'organic liquid', 'solid']
full_text_T = pd.Index(FULL_TEXT_SETTINGS['temperatures_C'], name='T_C')
full_text_phase_amounts = pd.DataFrame(np.nan, index=full_text_T, columns=full_text_phases)
full_text_phase_carbon = full_text_phase_amounts.copy()
full_text_compositions = {
    phase: pd.DataFrame(np.nan, index=full_text_T,
                         columns=FULL_TEXT_SETTINGS['phase_species'][phase])
    for phase in full_text_phases}
for result in full_text_results:
    t = result['temperature_C']
    total_feed_C = result['feed_atoms'][result['elements'].index('C')]
    for phase in full_text_phases:
        names = FULL_TEXT_SETTINGS['phase_species'][phase]
        amounts = np.array([result['amounts'].get((phase, name), 0.0) for name in names])
        registry = model.SOLID_DATA if phase == 'solid' else {
            **model.MOLECULAR_SPECIES, **model.ORGANIC_SPECIES, **model.ION_DATA}
        full_text_phase_amounts.loc[t, phase] = amounts.sum()
        full_text_phase_carbon.loc[t, phase] = sum(
            amount * registry[name]['elements'].get('C', 0.0)
            for name, amount in zip(names, amounts)) / total_feed_C
        if amounts.sum() > 1e-12:
            full_text_compositions[phase].loc[t] = amounts / amounts.sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
for ax, frame, title, ylabel in [
    (axes[0], full_text_phase_amounts, 'Phase amounts', 'mol/mol dry-feed C'),
    (axes[1], full_text_phase_carbon, 'Carbon allocation', 'Fraction of total feed carbon')]:
    for phase in full_text_phases:
        ax.plot(frame.index, frame[phase], 'o-', label=phase)
    ax.set(title=title, xlabel='Temperature [°C]', ylabel=ylabel,
           xlim=(full_text_T.min(), full_text_T.max()))
    ax.grid(alpha=.25); ax.legend()
axes[1].set_ylim(0, 1.05)
fig.suptitle(f'full_text: reduced Figures 7–8 feed + selected ions — {FULL_TEXT_SETTINGS["pressure_bar"]:g} bar')
plt.show()
fig, axes = plt.subplots(2, 2, figsize=(15, 10), layout='constrained')
for ax, phase in zip(axes.flat, full_text_phases):
    for name in full_text_compositions[phase].columns:
        ax.plot(full_text_T, full_text_compositions[phase][name], 'o-', ms=3, label=name)
    ax.set(title=phase, xlabel='Temperature [°C]', ylabel='Species mole fraction in phase',
           ylim=(0, 1.05), xlim=(full_text_T.min(), full_text_T.max()))
    ax.grid(alpha=.25)
    if len(full_text_compositions[phase].columns): ax.legend(fontsize=7)
    if not full_text_compositions[phase].notna().any().any():
        ax.text(.5, .5, 'Phase absent or no solved states', ha='center', va='center',
                transform=ax.transAxes)
fig.suptitle('Phase compositions including water; solids use formula-unit fractions')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rows = []

for result in full_text_results:
    # Convert mol/mol dry-feed C to mol/kg dry feed.
    factor = result["feed_C_mol_per_g_dry"] * 1000.0

    for (phase, species), amount in result["amounts"].items():
        if phase != "solid":
            continue

        elements = model.SOLID_DATA[species]["elements"]
        molecular_weight = sum(
            model.AW[element] * count
            for element, count in elements.items()
        )

        rows.append({
            "T_C": result["temperature_C"],
            "Species": species,
            "Solid_g_per_kg_dry_feed":
                amount * factor * molecular_weight,
        })

solid_yields = pd.DataFrame(rows)

if solid_yields.empty:
    print("No solid results available. Run the full_text sweep first.")
else:
    temperatures = sorted(set(FULL_TEXT_SETTINGS["temperatures_C"]))
    solved_temperatures = {
        result["temperature_C"] for result in full_text_results
    }

    table = solid_yields.pivot_table(
        index="T_C",
        columns="Species",
        values="Solid_g_per_kg_dry_feed",
        aggfunc="sum",
    ).reindex(temperatures)

    # Missing species at successful points mean zero.
    # Failed temperatures remain NaN, producing gaps.
    for temperature in solved_temperatures:
        table.loc[temperature] = table.loc[temperature].fillna(0.0)

    fig, axes = plt.subplots(
        1, 2, figsize=(13, 5), sharex=True,
        layout="constrained"
    )

    groups = [
        ("Char", ["carbon"]),
        ("Minerals and hydrates", [
            species for species in table.columns
            if species != "carbon"
        ]),
    ]

    for ax, (title, species_list) in zip(axes, groups):
        plotted = False

        for species in species_list:
            if species not in table.columns:
                continue

            values = table[species]
            if values.max(skipna=True) > 1e-6:
                ax.plot(
                    table.index, values,
                    marker="o", markersize=4, label=species
                )
                plotted = True

        cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")
        if cutoff is not None:
            ax.axvline(
                cutoff, color="black", linestyle="--",
                linewidth=1, alpha=0.6
            )

        ax.set_title(title)
        ax.set_xlabel("Temperature [°C]")
        ax.set_ylabel("Solid yield [g/kg dry feed]")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)

        if plotted:
            ax.legend(fontsize=8)
        else:
            ax.text(
                0.5, 0.5, "No solids above 10⁻⁶ g/kg",
                transform=ax.transAxes,
                ha="center", va="center"
            )

    fig.suptitle(
        "Predicted equilibrium solids\n"
        "Dashed line: aqueous cut-off; gaps: failed calculations"
    )
    plt.show()

    print("Solid yields [g/kg dry feed]:")
    display(table.round(6))

5. Controlled gas-carbon tests: at least 90% versus unrestricted
These two sweeps clone the same FULL_TEXT_SETTINGS. Both remove the exact gas-carbon equality; the first adds a >=90% inequality and the second adds no gas-carbon constraint. Feed, selected ions/minerals, pressure, cut-off, temperature grid and solver controls are otherwise identical. Any gas entry in phase_carbon_fractions is removed from both tests to avoid a hidden equality. Other optional constraints are retained.
Run the loading cell with the updated model, then the full_text inputs cell, then this test cell. You do not need to rerun the earlier comparison sweeps. Results are stored separately in carbon_test_results and carbon_test_summary. Gas yields are mol/kg dry feed; the summary additionally reports dry gas mole percentages (water excluded). Unrestricted equilibrium may put more OR less than 90% of carbon in the gas. A lower bound of 90% does not force an exact 90%.

In [ ]:
# TWO CONTROLLED TESTS: SAME FEED, SPECIES, PRESSURE AND AQUEOUS CUT-OFF.
# Run the model-loading cell and full_text input cell first.
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if not getattr(model, 'SUPPORTS_GAS_CARBON_MIN_FRACTION', False):
    raise RuntimeError('Reload the UPDATED HTG_Model(2).ipynb using the loading cell, then rerun the full_text inputs cell.')

CARBON_TESTS = {
    'Gas carbon >=90%': 0.90,
    'Gas carbon unrestricted': None,
}
carbon_test_results = {label: [] for label in CARBON_TESTS}
carbon_test_rows = []
registry = {**model.ALL_MOLECULAR_DATA, **model.SOLID_DATA, **model.ION_DATA}
phases = ['gas', 'aqueous', 'organic liquid', 'solid']
gas_names = ['carbon dioxide', 'methane', 'hydrogen', 'carbon monoxide']

for label, minimum in CARBON_TESTS.items():
    for temperature in FULL_TEXT_SETTINGS['temperatures_C']:
        case = copy.deepcopy(FULL_TEXT_SETTINGS)
        case['temperature_C'] = float(temperature)
        case['gas_carbon_fraction'] = None  # Remove the previous EXACT 90% equality.
        case['gas_carbon_min_fraction'] = minimum
        # Clear a duplicate gas equality if one was manually added here.
        case['phase_carbon_fractions'] = {
            p: f for p, f in case.get('phase_carbon_fractions', {}).items() if p != 'gas'
        }
        mode = FULL_TEXT_INITIALIZATION[float(temperature)]
        case['liquid_initialization'] = mode == 'liquid'
        case['gas_water_initialization'] = mode == 'vapor'
        cutoff = case.get('aqueous_cutoff_C')
        if case.get('aqueous_enabled') is False or (cutoff is not None and temperature >= cutoff):
            case['liquid_initialization'] = False
            case['gas_water_initialization'] = True
        row = {'Test':label, 'T_C':float(temperature), 'Status':'failed',
               'Error':'', 'Gas_C_percent':np.nan, 'Char_g_per_kg':np.nan,
               'G_J_per_mol_dry_C':np.nan, 'Balance_residual':np.nan,
               **{f'{p}_C_percent':np.nan for p in phases},
               **{f'{s}_mol_per_kg':np.nan for s in gas_names},
               **{f'{s}_dry_gas_mol_percent':np.nan for s in gas_names}}
        try:
            result = model.solve_equilibrium(case)
            carbon_test_results[label].append(result)
            amounts = result['amounts']
            feed_C = result['feed_atoms'][result['elements'].index('C')]
            mol_per_kg_factor = result['feed_C_mol_per_g_dry'] * 1000.
            for phase in phases:
                carbon = sum(n*registry[s]['elements'].get('C', 0)
                             for (p,s),n in amounts.items() if p == phase)
                row[f'{phase}_C_percent'] = 100.*carbon/feed_C
            dry_gas = sum(n for (p,s),n in amounts.items() if p == 'gas' and s != 'water')
            for species in gas_names:
                amount = amounts.get(('gas', species), 0.)
                row[f'{species}_mol_per_kg'] = amount*mol_per_kg_factor
                row[f'{species}_dry_gas_mol_percent'] = 100.*amount/dry_gas if dry_gas > 0 else np.nan
            row.update(Status='solved', Gas_C_percent=100.*result['gas_carbon_fraction_actual'],
                Char_g_per_kg=amounts.get(('solid','carbon'),0.)*mol_per_kg_factor*model.AW['C'],
                G_J_per_mol_dry_C=result['gibbs_J_per_mol_feed_C'],
                Balance_residual=result['balance_residual'])
        except Exception as exc:
            row['Error'] = str(exc)
        carbon_test_rows.append(row)
        print(f'{label}; {temperature:g} C: {row["Status"]}')

carbon_test_summary = pd.DataFrame(carbon_test_rows)
display(carbon_test_summary)
# A lower Gibbs energy is necessary for the unrestricted optimum when both
# tests converge to the global optimum. A higher value flags a convergence issue.
energy_check = carbon_test_summary.pivot(index='T_C', columns='Test', values='G_J_per_mol_dry_C')
energy_check['Unrestricted_minus_min90_J_per_mol_dry_C'] = (
    energy_check['Gas carbon unrestricted']-energy_check['Gas carbon >=90%'])
print('Positive energy differences above numerical tolerance warrant convergence checks:')
display(energy_check)

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, layout='constrained')
plot_specs = [
    ('Gas_C_percent', 'Feed carbon in gas', 'Carbon [% of total feed C]'),
    ('Char_g_per_kg', 'Char yield', 'g/kg dry feed'),
    ('carbon dioxide_mol_per_kg', 'CO2 yield', 'mol/kg dry feed'),
    ('methane_mol_per_kg', 'CH4 yield', 'mol/kg dry feed'),
    ('hydrogen_mol_per_kg', 'H2 yield', 'mol/kg dry feed'),
    ('carbon monoxide_mol_per_kg', 'CO yield', 'mol/kg dry feed'),
]
for ax, (column, title, unit) in zip(axes.flat, plot_specs):
    for label in CARBON_TESTS:
        table = carbon_test_summary[carbon_test_summary.Test.eq(label)].set_index('T_C')
        ax.plot(table.index, table[column], marker='o', markersize=3, label=label)
    cutoff = FULL_TEXT_SETTINGS.get('aqueous_cutoff_C')
    if cutoff is not None:
        ax.axvline(cutoff, color='black', linestyle='--', alpha=.5, linewidth=1)
    ax.set_title(title)
    ax.set_ylabel(unit)
    ax.set_xlabel('Temperature [C]')
    ax.grid(alpha=.25)
    ax.set_ylim(bottom=0)
axes[0,0].axhline(90., color='grey', linestyle=':', linewidth=1)
axes[0,0].legend(fontsize=8)
fig.suptitle('Minimum 90% gas carbon vs unrestricted gas carbon\nSame feed and products; dashed line = aqueous cut-off; gaps = failed points')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LABEL = "Gas carbon unrestricted"
TOP_SPECIES = 10
MIN_YIELD = 1e-8  # mol/kg dry feed

results = carbon_test_results.get(LABEL, [])
if not results:
    raise RuntimeError("No successful unrestricted results. Run Section 5 first.")
if "paper78" not in globals():
    raise RuntimeError("Run the paper78 reference-data cell first.")

temperatures = sorted(set(FULL_TEXT_SETTINGS["temperatures_C"]))
cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")
solved = {r["temperature_C"] for r in results}

# Convert all species amounts to mol/kg dry feed.
rows, comparison_rows = [], []
gas_map = {
    "CO2": "carbon dioxide",
    "CH4": "methane",
    "H2": "hydrogen",
    "N2": "nitrogen",
    "NH3": "ammonia",
    "H2S": "hydrogen sulfide",
    "CO": "carbon monoxide",
}

for result in results:
    temperature = result["temperature_C"]
    factor = result["feed_C_mol_per_g_dry"] * 1000.0
    yields = {
        pair: amount * factor
        for pair, amount in result["amounts"].items()
    }

    for (phase, species), value in yields.items():
        group = phase
        if phase == "aqueous":
            group = (
                "aqueous ions" if species in model.ION_DATA
                else "aqueous neutrals"
            )
        rows.append({
            "T_C": temperature,
            "Group": group,
            "Species": species,
            "Yield": value,
        })

    comparison_rows.append({
        "T_C": temperature,
        **{
            symbol: yields.get(("gas", species), 0.0)
            for symbol, species in gas_map.items()
        },
        "H2O_g": yields.get(("gas", "water"), 0.0),
        "H2O_l": sum(
            yields.get((phase, "water"), 0.0)
            for phase in ["aqueous", "organic liquid"]
        ),
    })

unrestricted_species = pd.DataFrame(rows)
unrestricted_fig78 = (
    pd.DataFrame(comparison_rows)
    .set_index("T_C")
    .reindex(temperatures)
)

def format_axis(ax, title, ylabel):
    ax.set_title(title)
    ax.set_xlabel("Temperature [°C]")
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    if cutoff is not None:
        ax.axvline(
            cutoff, color="black", linestyle="--",
            linewidth=1, alpha=0.5
        )

# ------------------------------------------------------------
# 1. Main species in every phase, including aqueous ions
# ------------------------------------------------------------
fig, axes = plt.subplots(
    3, 2, figsize=(15, 13), layout="constrained"
)

groups = [
    ("gas", "Gas / supercritical fluid"),
    ("aqueous neutrals", "Aqueous neutral species"),
    ("aqueous ions", "Aqueous ions"),
    ("organic liquid", "Organic liquid"),
    ("solid", "Solids"),
]

for ax, (group, title) in zip(axes.flat, groups):
    subset = unrestricted_species[
        unrestricted_species["Group"].eq(group)
    ]

    if not subset.empty:
        table = subset.pivot_table(
            index="T_C", columns="Species",
            values="Yield", aggfunc="sum"
        ).reindex(temperatures)

        # Absent species at solved points are zero.
        # Failed temperatures remain NaN and appear as gaps.
        for temperature in solved:
            table.loc[temperature] = (
                table.loc[temperature].fillna(0.0)
            )

        ranking = (
            table.max()
            .drop(labels=["water"], errors="ignore")
            .sort_values(ascending=False)
        )
        selected = ranking[ranking > MIN_YIELD].head(TOP_SPECIES).index

        for species in selected:
            ax.plot(
                table.index, table[species],
                "-o", markersize=3, label=(
    model.SOLID_DATA[species].get("formula", species)
    if group == "solid"
    else species
)
            )

        if len(selected):
            ax.set_yscale("symlog", linthresh=MIN_YIELD)
            ax.legend(fontsize=8)

    if not ax.get_legend_handles_labels()[0]:
        ax.text(
            0.5, 0.5, "No species above reporting threshold",
            transform=ax.transAxes, ha="center", va="center"
        )

    format_axis(ax, title, "Species yield [mol/kg dry feed]")

ax = axes.flat[5]
ax.plot(
    unrestricted_fig78.index, unrestricted_fig78["H2O_g"],
    "-o", markersize=3, label="Gas water"
)
ax.plot(
    unrestricted_fig78.index, unrestricted_fig78["H2O_l"],
    "-s", markersize=3, label="Liquid-phase water"
)
format_axis(ax, "Water distribution", "Water [mol/kg dry feed]")
ax.legend()

fig.suptitle(
    "Unrestricted gas-carbon case: species in all phases\n"
    "Water shown separately; species panels use symmetric-log scales"
)
plt.show()

# ------------------------------------------------------------
# 2. Figure 7 comparison: major and minor gas species
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 2, figsize=(15, 5), layout="constrained"
)

for ax, columns, title in [
    (axes[0], ["CO2", "CH4", "H2"], "Figure 7: major gases"),
    (axes[1], ["N2", "NH3", "H2S", "CO"], "Figure 7: minor gases"),
]:
    for column in columns:
        line, = ax.plot(
            unrestricted_fig78.index, unrestricted_fig78[column],
            "-", linewidth=1.8, label=f"{column}: unrestricted"
        )
        ax.plot(
            paper78["T_C"], paper78[column],
            "o", markersize=4, markerfacecolor="none",
            color=line.get_color(), label=f"{column}: reference"
        )

    format_axis(ax, title, "Gas yield [mol/kg dry feed]")
    ax.legend(fontsize=8)

fig.suptitle(
    "Unrestricted case versus Figure 7 reference data\n"
    "Lines: current model; circles: notebook digitization"
)
plt.show()

# ------------------------------------------------------------
# 3. Figure 8 comparison: gaseous and liquid water
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")

for column, name in [
    ("H2O_g", "Gas water"),
    ("H2O_l", "Liquid-phase water"),
]:
    line, = ax.plot(
        unrestricted_fig78.index, unrestricted_fig78[column],
        "-", linewidth=1.8, label=f"{name}: unrestricted"
    )
    ax.plot(
        paper78["T_C"], paper78[column],
        "o", markersize=4, markerfacecolor="none",
        color=line.get_color(), label=f"{name}: reference"
    )

format_axis(ax, "Figure 8: water distribution", "Water [mol/kg dry feed]")
ax.legend(fontsize=9)
plt.show()

extended species list

In [ ]:
from pathlib import Path
import json
import types
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_NOTEBOOK = Path("HTG_Model(2).ipynb")
DATABASE_DIRECTORY = Path(".")


def locate_file(preferred_names):
    for folder in (
        DATABASE_DIRECTORY,
        DATABASE_DIRECTORY / "upload",
    ):
        for name in preferred_names:
            candidate = folder / name
            if candidate.is_file():
                return candidate

    raise FileNotFoundError(
        f"File not found; set an explicit path: {preferred_names}"
    )


DATABASE_PATHS = {
    "nasa": locate_file([
        "Thermdat_WasteMech.txt",
        "Thermdat_WasteMech(3).txt",
    ]),
    "pcsaft": locate_file([
        "PROJECT_PCSAFT.csv",
        "PROJECT_PCSAFT(4).csv",
    ]),
    "epcsaft": locate_file([
        "PROJECT_EPCSAFT.csv",
        "PROJECT_EPCSAFT(5).csv",
    ]),
    "supcrt": locate_file([
        "supcrt92(7).xml",
        "supcrt92.xml",
    ]),
}


def load_model_notebook(path):
    document = json.loads(
        Path(path).read_text(encoding="utf-8")
    )
    module = types.ModuleType("htg_model")

    for index, cell in enumerate(document["cells"]):
        if cell["cell_type"] == "code":
            source = "".join(cell["source"])
            exec(
                compile(
                    source,
                    f"{path}:cell{index + 1}",
                    "exec",
                ),
                module.__dict__,
            )

    return module


# Acetone and propanal have the same elemental formula.
# Explicitly select the acetone NASA record.
SPECIES_OVERRIDES = {
    "acetone": {"nasa": "CH3COCH3"},
}

print("Loading model:", MODEL_NOTEBOOK.resolve())
model = load_model_notebook(MODEL_NOTEBOOK)

if not getattr(
    model, "SUPPORTS_GAS_CARBON_MIN_FRACTION", False
):
    raise RuntimeError(
        "This model file does not support the gas-carbon "
        "inequality. Select the updated model notebook."
    )

available = model.initialize_model(
    DATABASE_PATHS,
    species_overrides=SPECIES_OVERRIDES,
)

species_catalogue = model.available_species(
    include_unavailable=True
)

display(available)

In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if not getattr(
    model, "SUPPORTS_GAS_CARBON_MIN_FRACTION", False
):
    raise RuntimeError(
        "Run the updated model-loading cell first."
    )

# ============================================================
# COMMON SPECIES SETTINGS FOR BOTH TESTS
# ============================================================

CARBON_TEST_LIQUID_SPECIES = [
    "benzene",
    "toluene",
    "methanol",
    "ethanol",
    "acetone",
]

CARBON_TEST_PHASE_SPECIES = copy.deepcopy(
    FULL_TEXT_SETTINGS["phase_species"]
)

# Permit these molecules to partition between fluid phases.
for phase in ["gas", "aqueous"]:
    CARBON_TEST_PHASE_SPECIES[phase] = list(dict.fromkeys(
        CARBON_TEST_PHASE_SPECIES.get(phase, [])
        + CARBON_TEST_LIQUID_SPECIES
    ))

# Add dissolved N2 below the aqueous cut-off.
CARBON_TEST_PHASE_SPECIES["aqueous"] = list(dict.fromkeys(
    CARBON_TEST_PHASE_SPECIES["aqueous"] + ["nitrogen"]
))

CARBON_TEST_PHASE_SPECIES["organic liquid"] = list(
    CARBON_TEST_LIQUID_SPECIES
)

# ============================================================
# THERMOCHEMISTRY AND SAFT COVERAGE CHECKS
# ============================================================

if (
    model.ALL_MOLECULAR_DATA.get("acetone", {}).get("nasa")
    != "CH3COCH3"
):
    raise RuntimeError(
        "Acetone is not mapped to CH3COCH3. "
        "Rerun Cell 1 with the explicit species override."
    )

coverage = []

for species in CARBON_TEST_LIQUID_SPECIES + ["nitrogen"]:
    data = model.ALL_MOLECULAR_DATA[species]
    nasa_name = data["nasa"]

    for temperature in FULL_TEXT_SETTINGS["temperatures_C"]:
        temperature_K = float(temperature) + 273.15

        model.resolve_molecular_parameters(
            temperature_K, [species]
        )
        model.nasa7_properties(
            model.nasa_db, nasa_name, temperature_K
        )

    coverage.append({
        "Species": species,
        "Formula": data["formula"],
        "NASA": nasa_name,
        "Temperature_grid_covered": True,
    })

print("Common liquid candidates and dissolved nitrogen:")
display(pd.DataFrame(coverage))

print("Common phase-species selections:")
for phase, names in CARBON_TEST_PHASE_SPECIES.items():
    print(f"{phase}: {names}")

# ============================================================
# TWO TEST CASES
# ============================================================

CARBON_TESTS = {
    "Gas carbon >=90%": 0.90,
    "Gas carbon unrestricted": None,
}

carbon_test_results = {
    label: [] for label in CARBON_TESTS
}
carbon_test_rows = []

registry = {
    **model.ALL_MOLECULAR_DATA,
    **model.SOLID_DATA,
    **model.ION_DATA,
}

phases = [
    "gas",
    "aqueous",
    "organic liquid",
    "solid",
]

gas_names = [
    "carbon dioxide",
    "methane",
    "hydrogen",
    "carbon monoxide",
]

for label, minimum in CARBON_TESTS.items():
    for temperature in FULL_TEXT_SETTINGS["temperatures_C"]:
        case = copy.deepcopy(FULL_TEXT_SETTINGS)

        case["phase_species"] = copy.deepcopy(
            CARBON_TEST_PHASE_SPECIES
        )
        case["temperature_C"] = float(temperature)

        # Remove the previous EXACT gas-carbon equality.
        case["gas_carbon_fraction"] = None

        # First test: >=90%; second test: no gas-carbon bound.
        case["gas_carbon_min_fraction"] = minimum

        # Remove a duplicate gas equality if manually added.
        case["phase_carbon_fractions"] = {
            phase: fraction
            for phase, fraction in case.get(
                "phase_carbon_fractions", {}
            ).items()
            if phase != "gas"
        }

        mode = FULL_TEXT_INITIALIZATION[float(temperature)]
        case["liquid_initialization"] = mode == "liquid"
        case["gas_water_initialization"] = mode == "vapor"

        cutoff = case.get("aqueous_cutoff_C")
        aqueous_disabled = (
            case.get("aqueous_enabled") is False
            or (
                cutoff is not None
                and temperature >= cutoff
            )
        )

        if aqueous_disabled:
            case["liquid_initialization"] = False
            case["gas_water_initialization"] = True

        row = {
            "Test": label,
            "T_C": float(temperature),
            "Status": "failed",
            "Error": "",
            "Gas_C_percent": np.nan,
            "Char_g_per_kg": np.nan,
            "G_J_per_mol_dry_C": np.nan,
            "Balance_residual": np.nan,
            **{
                f"{phase}_C_percent": np.nan
                for phase in phases
            },
            **{
                f"{species}_mol_per_kg": np.nan
                for species in gas_names
            },
            **{
                f"{species}_dry_gas_mol_percent": np.nan
                for species in gas_names
            },
        }

        try:
            result = model.solve_equilibrium(case)
            carbon_test_results[label].append(result)

            amounts = result["amounts"]
            feed_C = result["feed_atoms"][
                result["elements"].index("C")
            ]

            factor = (
                result["feed_C_mol_per_g_dry"] * 1000.0
            )

            # Carbon allocation among all phases.
            for phase in phases:
                carbon = sum(
                    amount
                    * registry[species]["elements"].get("C", 0)
                    for (p, species), amount in amounts.items()
                    if p == phase
                )
                row[f"{phase}_C_percent"] = (
                    100.0 * carbon / feed_C
                )

            # Dry gas composition excludes molecular water.
            dry_gas = sum(
                amount
                for (phase, species), amount in amounts.items()
                if phase == "gas" and species != "water"
            )

            for species in gas_names:
                amount = amounts.get(("gas", species), 0.0)

                row[f"{species}_mol_per_kg"] = (
                    amount * factor
                )
                row[f"{species}_dry_gas_mol_percent"] = (
                    100.0 * amount / dry_gas
                    if dry_gas > 0 else np.nan
                )

            row.update(
                Status="solved",
                Gas_C_percent=(
                    100.0
                    * result["gas_carbon_fraction_actual"]
                ),
                Char_g_per_kg=(
                    amounts.get(("solid", "carbon"), 0.0)
                    * factor
                    * model.AW["C"]
                ),
                G_J_per_mol_dry_C=(
                    result["gibbs_J_per_mol_feed_C"]
                ),
                Balance_residual=result["balance_residual"],
            )

        except Exception as exc:
            row["Error"] = str(exc)

        carbon_test_rows.append(row)
        print(
            f"{label}; {temperature:g} °C: "
            f"{row['Status']}"
        )

carbon_test_summary = pd.DataFrame(carbon_test_rows)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(carbon_test_summary)

# ============================================================
# GIBBS-ENERGY COMPARISON
# ============================================================

energy_check = carbon_test_summary.pivot(
    index="T_C",
    columns="Test",
    values="G_J_per_mol_dry_C",
)

energy_check[
    "Unrestricted_minus_min90_J_per_mol_dry_C"
] = (
    energy_check["Gas carbon unrestricted"]
    - energy_check["Gas carbon >=90%"]
)

print(
    "The unrestricted global optimum should have equal "
    "or lower Gibbs energy. Positive differences beyond "
    "numerical tolerance warrant convergence checks."
)
display(energy_check)

# ============================================================
# COMPARISON PLOTS
# ============================================================

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 9),
    sharex=True,
    layout="constrained",
)

plot_specs = [
    (
        "Gas_C_percent",
        "Feed carbon in gas",
        "Carbon [% of total feed C]",
    ),
    (
        "Char_g_per_kg",
        "Char yield",
        "g/kg dry feed",
    ),
    (
        "carbon dioxide_mol_per_kg",
        "CO₂ yield",
        "mol/kg dry feed",
    ),
    (
        "methane_mol_per_kg",
        "CH₄ yield",
        "mol/kg dry feed",
    ),
    (
        "hydrogen_mol_per_kg",
        "H₂ yield",
        "mol/kg dry feed",
    ),
    (
        "carbon monoxide_mol_per_kg",
        "CO yield",
        "mol/kg dry feed",
    ),
]

for ax, (column, title, unit) in zip(
    axes.flat, plot_specs
):
    for label in CARBON_TESTS:
        table = (
            carbon_test_summary[
                carbon_test_summary["Test"].eq(label)
            ]
            .set_index("T_C")
            .sort_index()
        )

        ax.plot(
            table.index,
            table[column],
            marker="o",
            markersize=3,
            label=label,
        )

    cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")
    if cutoff is not None:
        ax.axvline(
            cutoff,
            color="black",
            linestyle="--",
            alpha=0.5,
            linewidth=1,
        )

    ax.set_title(title)
    ax.set_ylabel(unit)
    ax.set_xlabel("Temperature [°C]")
    ax.grid(alpha=0.25)
    ax.set_ylim(bottom=0)

axes[0, 0].axhline(
    90.0, color="grey", linestyle=":", linewidth=1
)
axes[0, 0].legend(fontsize=8)

fig.suptitle(
    "Minimum 90% gas carbon versus unrestricted gas carbon\n"
    "Both include aqueous N₂ and expanded liquid candidates; "
    "dashed line = aqueous cut-off"
)

plt.show()

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOP_SPECIES = 12
MIN_YIELD = 1e-8  # mol/kg dry feed

registry = {
    **model.ALL_MOLECULAR_DATA,
    **model.SOLID_DATA,
    **model.ION_DATA,
}


def formula_label(species):
    """Use database formulas, with explicit charges for aqueous ions."""
    data = registry[species]
    formula = data.get("formula", species)

    if species in model.ION_DATA:
        charge = int(data["charge"])
        # Remove the trailing charge notation, preserving atom counts.
        formula = re.sub(r"\^?\d*[+-]$", "", formula)
        # Species identifiers retain formulas more reliably for
        # monatomic ions such as Mg2+.
        elements = data["elements"]
        if len(elements) == 1 and next(iter(elements.values())) == 1:
            formula = next(iter(elements))

        magnitude = str(abs(charge)) if abs(charge) > 1 else ""
        sign = "+" if charge > 0 else "−"
        return f"{formula} ({magnitude}{sign})"

    return formula.replace("*", "·")


panels = [
    ("gas", "Gas / supercritical fluid"),
    ("aqueous neutrals", "Aqueous neutral species"),
    ("aqueous ions", "Aqueous ions"),
    ("organic liquid", "Organic liquid"),
    ("solid", "Solids"),
]

temperatures = sorted(set(FULL_TEXT_SETTINGS["temperatures_C"]))
cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")
phase_formula_tables = {}

for test, results in carbon_test_results.items():
    if not results:
        print(f"{test}: no successful results.")
        continue

    rows = []
    solved = {r["temperature_C"] for r in results}

    for result in results:
        factor = result["feed_C_mol_per_g_dry"] * 1000.0

        for (phase, species), amount in result["amounts"].items():
            if phase == "aqueous":
                group = (
                    "aqueous ions"
                    if species in model.ION_DATA
                    else "aqueous neutrals"
                )
            else:
                group = phase

            rows.append({
                "T_C": result["temperature_C"],
                "Group": group,
                "Species": species,
                "Yield": amount * factor,
            })

    data = pd.DataFrame(rows)
    phase_formula_tables[test] = {}

    fig, axes = plt.subplots(
        3, 2, figsize=(15, 13), layout="constrained"
    )

    for ax, (group, title) in zip(axes.flat, panels):
        subset = data[data["Group"].eq(group)]

        if subset.empty:
            table = pd.DataFrame(index=temperatures)
        else:
            table = subset.pivot_table(
                index="T_C",
                columns="Species",
                values="Yield",
                aggfunc="sum",
            ).reindex(temperatures)

            # Successful points: absent species = zero.
            # Failed points: retain NaN so curves have gaps.
            for temperature in solved:
                table.loc[temperature] = (
                    table.loc[temperature].fillna(0.0)
                )

        phase_formula_tables[test][group] = table

        ranking = (
            table.max()
            .drop(labels=["water"], errors="ignore")
            .sort_values(ascending=False)
        )
        selected = ranking[ranking > MIN_YIELD].head(TOP_SPECIES).index

        # Disambiguate different species sharing one molecular formula.
        labels = [formula_label(s) for s in selected]
        for species, label in zip(selected, labels):
            if labels.count(label) > 1:
                label = f"{label} [{species}]"

            ax.plot(
                table.index,
                table[species],
                "-o",
                markersize=3,
                label=label,
            )

        if len(selected):
            ax.set_yscale("symlog", linthresh=MIN_YIELD)
            ax.legend(fontsize=8)
        else:
            ax.text(
                0.5, 0.5,
                "No non-water species above threshold",
                transform=ax.transAxes,
                ha="center", va="center",
            )

        ax.set_title(title)
        ax.set_xlabel("Temperature [°C]")
        ax.set_ylabel("Species yield [mol/kg dry feed]")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)

        if cutoff is not None:
            ax.axvline(
                cutoff, color="black", linestyle="--",
                linewidth=1, alpha=0.5,
            )

    # Sixth panel: molecular water in each fluid phase.
    ax = axes.flat[5]
    for group, label in [
        ("gas", "H₂O — gas"),
        ("aqueous neutrals", "H₂O — aqueous"),
        ("organic liquid", "H₂O — organic liquid"),
    ]:
        table = phase_formula_tables[test].get(group)
        if table is not None and "water" in table.columns:
            ax.plot(
                table.index, table["water"],
                "-o", markersize=3, label=label,
            )

    ax.set_title("Water distribution")
    ax.set_xlabel("Temperature [°C]")
    ax.set_ylabel("H₂O [mol/kg dry feed]")
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)

    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)

    if cutoff is not None:
        ax.axvline(
            cutoff, color="black", linestyle="--",
            linewidth=1, alpha=0.5,
        )

    fig.suptitle(
        f"{test}: species distribution by phase\n"
        "Formula labels; water shown separately; "
        "species panels use symmetric-log scales"
    )
    plt.show()



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "paper78" not in globals():
    raise RuntimeError("Run the paper78 reference-data cell first.")

gas_map = {
    "CO2": "carbon dioxide",
    "CH4": "methane",
    "H2": "hydrogen",
    "N2": "nitrogen",
    "NH3": "ammonia",
    "H2S": "hydrogen sulfide",
    "CO": "carbon monoxide",
}
formula_labels = {
    "CO2": "CO₂",
    "CH4": "CH₄",
    "H2": "H₂",
    "N2": "N₂",
    "NH3": "NH₃",
    "H2S": "H₂S",
    "CO": "CO",
    "H2O_g": "H₂O — gas",
    "H2O_l": "H₂O — liquid",
}

temperatures = sorted(set(FULL_TEXT_SETTINGS["temperatures_C"]))
cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")
fig78_comparisons = {}

for test, results in carbon_test_results.items():
    if not results:
        print(f"{test}: no successful results.")
        continue

    rows = []
    for result in results:
        factor = result["feed_C_mol_per_g_dry"] * 1000.0
        yields = {
            pair: amount * factor
            for pair, amount in result["amounts"].items()
        }

        rows.append({
            "T_C": result["temperature_C"],
            **{
                symbol: yields.get(("gas", species), 0.0)
                for symbol, species in gas_map.items()
            },
            "H2O_g": yields.get(("gas", "water"), 0.0),
            "H2O_l": sum(
                yields.get((phase, "water"), 0.0)
                for phase in ["aqueous", "organic liquid"]
            ),
        })

    calculated = (
        pd.DataFrame(rows)
        .set_index("T_C")
        .reindex(temperatures)
    )
    fig78_comparisons[test] = calculated

    # Figure 7: major and minor gas-species yields.
    fig, axes = plt.subplots(
        1, 2, figsize=(15, 5), layout="constrained"
    )

    for ax, columns, title in [
        (axes[0], ["CO2", "CH4", "H2"], "Figure 7: major gases"),
        (axes[1], ["N2", "NH3", "H2S", "CO"], "Figure 7: minor gases"),
    ]:
        for column in columns:
            label = formula_labels[column]
            line, = ax.plot(
                calculated.index, calculated[column],
                "-", linewidth=1.8,
                label=f"{label}: model",
            )
            ax.plot(
                paper78["T_C"], paper78[column],
                "o", markersize=4, markerfacecolor="none",
                color=line.get_color(),
                label=f"{label}: reference",
            )

        if cutoff is not None:
            ax.axvline(
                cutoff, color="black", linestyle="--",
                linewidth=1, alpha=0.5,
            )

        ax.set_title(title)
        ax.set_xlabel("Temperature [°C]")
        ax.set_ylabel("Gas yield [mol/kg dry feed]")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"{test} — Figure 7 comparison\n"
        "Lines: current model; circles: notebook reference digitization"
    )
    plt.show()

    # Figure 8: gaseous and liquid molecular water.
    fig, ax = plt.subplots(
        figsize=(9, 5), layout="constrained"
    )

    for column in ["H2O_g", "H2O_l"]:
        label = formula_labels[column]
        line, = ax.plot(
            calculated.index, calculated[column],
            "-", linewidth=1.8,
            label=f"{label}: model",
        )
        ax.plot(
            paper78["T_C"], paper78[column],
            "o", markersize=4, markerfacecolor="none",
            color=line.get_color(),
            label=f"{label}: reference",
        )

    if cutoff is not None:
        ax.axvline(
            cutoff, color="black", linestyle="--",
            linewidth=1, alpha=0.5,
        )

    ax.set_title(f"{test} — Figure 8: water distribution")
    ax.set_xlabel("Temperature [°C]")
    ax.set_ylabel("Water [mol/kg dry feed]")
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)
    plt.show()

solver based on previosu resulst

In [ ]:
import copy
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# COMMON SPECIES FOR BOTH TESTS
# ============================================================

LIQUID_SPECIES = [
    "benzene", "toluene", "methanol", "ethanol", "acetone"
]

CARBON_TEST_PHASE_SPECIES = copy.deepcopy(
    FULL_TEXT_SETTINGS["phase_species"]
)

for phase in ["gas", "aqueous"]:
    CARBON_TEST_PHASE_SPECIES[phase] = list(dict.fromkeys(
        CARBON_TEST_PHASE_SPECIES.get(phase, [])
        + LIQUID_SPECIES
    ))

CARBON_TEST_PHASE_SPECIES["aqueous"] = list(dict.fromkeys(
    CARBON_TEST_PHASE_SPECIES["aqueous"] + ["nitrogen"]
))

CARBON_TEST_PHASE_SPECIES["organic liquid"] = (
    LIQUID_SPECIES.copy()
)

if not getattr(
    model, "SUPPORTS_GAS_CARBON_MIN_FRACTION", False
):
    raise RuntimeError("Load the updated model first.")

if (
    model.ALL_MOLECULAR_DATA.get("acetone", {}).get("nasa")
    != "CH3COCH3"
):
    raise RuntimeError(
        "Initialize the model with the explicit acetone "
        "NASA override."
    )

# ============================================================
# TEMPERATURE GRID
# ============================================================

REFINE_STEP_C = 5.0

original_grid = sorted(set(map(
    float, FULL_TEXT_SETTINGS["temperatures_C"]
)))

left = max(360.0, min(original_grid))
right = min(400.0, max(original_grid))

extra_grid = (
    np.arange(left, right + 1e-8, REFINE_STEP_C)
    if right >= left else []
)

test_temperatures = sorted(
    set(original_grid) | set(map(float, extra_grid))
)

cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")

if (
    cutoff is not None
    and min(original_grid) <= cutoff <= max(original_grid)
):
    test_temperatures = sorted(
        set(test_temperatures) | {float(cutoff)}
    )

# Existing plotting cells read this grid.
FULL_TEXT_SETTINGS["temperatures_C"] = test_temperatures

# ============================================================
# RESULT STORAGE
# ============================================================

CARBON_TESTS = {
    "Gas carbon >=90%": 0.90,
    "Gas carbon unrestricted": None,
}

carbon_test_results = {
    label: [] for label in CARBON_TESTS
}

summary_rows = []
trial_rows = []

registry = {
    **model.ALL_MOLECULAR_DATA,
    **model.SOLID_DATA,
    **model.ION_DATA,
}

phases = ["gas", "aqueous", "organic liquid", "solid"]

gas_names = [
    "carbon dioxide",
    "methane",
    "hydrogen",
    "carbon monoxide",
]

# ============================================================
# CONTINUATION + INDEPENDENT-START SWEEPS
# ============================================================

for label, minimum in CARBON_TESTS.items():
    # Independent continuation history for each test.
    previous = None

    for temperature in test_temperatures:
        case = copy.deepcopy(FULL_TEXT_SETTINGS)

        case["temperature_C"] = temperature
        case["phase_species"] = copy.deepcopy(
            CARBON_TEST_PHASE_SPECIES
        )

        case["gas_carbon_fraction"] = None
        case["gas_carbon_min_fraction"] = minimum

        case["phase_carbon_fractions"] = {
            phase: fraction
            for phase, fraction in case.get(
                "phase_carbon_fractions", {}
            ).items()
            if phase != "gas"
        }

        case["initial_phase_amounts"] = None

        disabled = (
            case.get("aqueous_enabled") is False
            or (
                cutoff is not None
                and temperature >= cutoff
            )
        )

        # Preserve specified modes at original temperatures.
        # New intermediate points default to liquid below
        # the cut-off and vapor above it.
        mode = FULL_TEXT_INITIALIZATION.get(
            temperature,
            "vapor" if disabled else "liquid",
        )

        case["liquid_initialization"] = (
            not disabled and mode == "liquid"
        )
        case["gas_water_initialization"] = (
            disabled or mode == "vapor"
        )

        effective, _ = model.prepare_phase_allocation(case)

        signature = {
            (phase, species)
            for phase, names
            in effective["phase_species"].items()
            for species in names
        }

        # Always use the previous successful lower-temperature
        # solution when the phase/species selections are compatible.
        if (
            previous is not None
            and set(previous["amounts"]) == signature
        ):
            starts = [
                ("continuation", previous["amounts"])
            ]
        else:
        # First point, or aqueous removal at the cut-off:
        # generate a fresh balanced starting composition.
            starts = [
            ("independent LP", None)
        ]




        

        valid = []

        for start_name, initial in starts:
            trial = copy.deepcopy(case)
            trial["initial_phase_amounts"] = (
                copy.deepcopy(initial)
            )

            diagnostic = {
                "Test": label,
                "T_C": temperature,
                "Start": start_name,
                "Seed_T_C": (
                    previous["temperature_C"]
                    if initial is not None else np.nan
                ),
            }

            try:
                result = model.solve_equilibrium(trial)

                if (
                    initial is not None
                    and result["phase_allocation"].get(
                        "initial_composition_reset"
                    )
                ):
                    raise RuntimeError(
                        "Continuation guess was unexpectedly reset."
                    )

                if result["balance_residual"] > 1e-7:
                    raise RuntimeError(
                        "Equality-constraint residual too large."
                    )

                if abs(result["charge_residual"]) > 1e-7:
                    raise RuntimeError(
                        "Charge residual too large."
                    )

                if (
                    minimum is not None
                    and result["gas_carbon_fraction_actual"]
                    < minimum - 1e-7
                ):
                    raise RuntimeError(
                        "Minimum gas-carbon fraction violated."
                    )

                valid.append((start_name, result))

                diagnostic.update(
                    Status="solved",
                    Error="",
                    G=result["gibbs_J_per_mol_feed_C"],
                    Residual=result["balance_residual"],
                )

            except Exception as exc:
                diagnostic.update(
                    Status="failed",
                    Error=str(exc),
                    G=np.nan,
                    Residual=np.nan,
                )

            trial_rows.append(diagnostic)

        row = {
            "Test": label,
            "T_C": temperature,
            "Status": "failed",
            "Selected_start": "",
            "Gas_C_percent": np.nan,
            "Char_g_per_kg": np.nan,
            "G_J_per_mol_dry_C": np.nan,
            "Balance_residual": np.nan,
            **{
                f"{phase}_C_percent": np.nan
                for phase in phases
            },
            **{
                f"{species}_mol_per_kg": np.nan
                for species in gas_names
            },
            **{
                f"{species}_dry_gas_mol_percent": np.nan
                for species in gas_names
            },
        }

        if valid:
            # Select the lower-G valid solution.
            winner, best = min(
                valid,
                key=lambda item: item[1][
                    "gibbs_J_per_mol_feed_C"
                ],
            )

            previous = best
            carbon_test_results[label].append(best)

            amounts = best["amounts"]
            feed_C = best["feed_atoms"][
                best["elements"].index("C")
            ]
            factor = (
                best["feed_C_mol_per_g_dry"] * 1000.0
            )

            for phase in phases:
                phase_carbon = sum(
                    amount
                    * registry[species]["elements"].get("C", 0)
                    for (p, species), amount in amounts.items()
                    if p == phase
                )

                row[f"{phase}_C_percent"] = (
                    100.0 * phase_carbon / feed_C
                )

            dry_gas = sum(
                amount
                for (phase, species), amount in amounts.items()
                if phase == "gas" and species != "water"
            )

            for species in gas_names:
                amount = amounts.get(
                    ("gas", species), 0.0
                )

                row[f"{species}_mol_per_kg"] = (
                    amount * factor
                )

                row[
                    f"{species}_dry_gas_mol_percent"
                ] = (
                    100.0 * amount / dry_gas
                    if dry_gas > 0 else np.nan
                )

            row.update(
                Status="solved",
                Selected_start=winner,
                Gas_C_percent=(
                    100.0
                    * best["gas_carbon_fraction_actual"]
                ),
                Char_g_per_kg=(
                    amounts.get(("solid", "carbon"), 0.0)
                    * factor
                    * model.AW["C"]
                ),
                G_J_per_mol_dry_C=(
                    best["gibbs_J_per_mol_feed_C"]
                ),
                Balance_residual=best["balance_residual"],
            )

        summary_rows.append(row)

        print(
            f"{label}; {temperature:g} °C: "
            f"{row['Status']}; {row['Selected_start']}",
            flush=True,
        )

# ============================================================
# DIAGNOSTICS
# ============================================================

carbon_test_summary = pd.DataFrame(summary_rows)
carbon_test_trials = pd.DataFrame(trial_rows)

display(carbon_test_summary)

print("Individual attempts, including failed starts:")
with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(carbon_test_trials)

energy_check = carbon_test_summary.pivot(
    index="T_C",
    columns="Test",
    values="G_J_per_mol_dry_C",
)

energy_check[
    "Unrestricted_minus_min90_J_per_mol_dry_C"
] = (
    energy_check["Gas carbon unrestricted"]
    - energy_check["Gas carbon >=90%"]
)

print(
    "Positive differences beyond numerical tolerance "
    "indicate unresolved convergence."
)
display(energy_check)

In [ ]:
import copy
import numpy as np

# Avoid recursively wrapping the function when this cell is rerun.
if not hasattr(model, "_allocator_before_continuation"):
    model._allocator_before_continuation = (
        model.prepare_phase_allocation
    )


def prepare_with_continuation(
    settings,
    _original=model._allocator_before_continuation,
):
    request = copy.deepcopy(settings)
    proposed = request.get("initial_phase_amounts")

    # Apply the original phase-selection rules first.
    request["initial_phase_amounts"] = None
    case, report = _original(request)
    report["initial_composition_reset"] = False

    if proposed is not None:
        if any(
            not np.isfinite(value) or value < 0
            for value in proposed.values()
        ):
            raise ValueError("Invalid continuation amounts")

        selected = {
            (phase, species)
            for phase, names in case["phase_species"].items()
            for species in names
        }

        lost = [
            pair
            for pair, value in proposed.items()
            if pair not in selected and value > 0
        ]

        if lost:
            # Removing populated phases would lose atoms.
            # Request a fresh, balanced LP starting composition.
            case["initial_phase_amounts"] = None
            report["initial_composition_reset"] = True
            report["initial_reset_reason"] = (
                f"Populated species no longer selected: {lost}"
            )
        else:
            # Preserve compatible guesses.
            # Unselected zero entries contain no atoms.
            case["initial_phase_amounts"] = {
                pair: value
                for pair, value in proposed.items()
                if pair in selected
            }

    return case, report


model.prepare_phase_allocation = prepare_with_continuation

print(
    "Continuation enabled: compatible initial guesses "
    "are retained above the aqueous cut-off."
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "paper78" not in globals():
    raise RuntimeError("Run the paper78 reference-data cell first.")

gas_map = {
    "CO2": "carbon dioxide",
    "CH4": "methane",
    "H2": "hydrogen",
    "N2": "nitrogen",
    "NH3": "ammonia",
    "H2S": "hydrogen sulfide",
    "CO": "carbon monoxide",
}
formula_labels = {
    "CO2": "CO₂",
    "CH4": "CH₄",
    "H2": "H₂",
    "N2": "N₂",
    "NH3": "NH₃",
    "H2S": "H₂S",
    "CO": "CO",
    "H2O_g": "H₂O — gas",
    "H2O_l": "H₂O — liquid",
}

temperatures = sorted(set(FULL_TEXT_SETTINGS["temperatures_C"]))
cutoff = FULL_TEXT_SETTINGS.get("aqueous_cutoff_C")
fig78_comparisons = {}

for test, results in carbon_test_results.items():
    if not results:
        print(f"{test}: no successful results.")
        continue

    rows = []
    for result in results:
        factor = result["feed_C_mol_per_g_dry"] * 1000.0
        yields = {
            pair: amount * factor
            for pair, amount in result["amounts"].items()
        }

        rows.append({
            "T_C": result["temperature_C"],
            **{
                symbol: yields.get(("gas", species), 0.0)
                for symbol, species in gas_map.items()
            },
            "H2O_g": yields.get(("gas", "water"), 0.0),
            "H2O_l": sum(
                yields.get((phase, "water"), 0.0)
                for phase in ["aqueous", "organic liquid"]
            ),
        })

    calculated = (
        pd.DataFrame(rows)
        .set_index("T_C")
        .reindex(temperatures)
    )
    fig78_comparisons[test] = calculated

    # Figure 7: major and minor gas-species yields.
    fig, axes = plt.subplots(
        1, 2, figsize=(15, 5), layout="constrained"
    )

    for ax, columns, title in [
        (axes[0], ["CO2", "CH4", "H2"], "Figure 7: major gases"),
        (axes[1], ["N2", "NH3", "H2S", "CO"], "Figure 7: minor gases"),
    ]:
        for column in columns:
            label = formula_labels[column]
            line, = ax.plot(
                calculated.index, calculated[column],
                "-", linewidth=1.8,
                label=f"{label}: model",
            )
            ax.plot(
                paper78["T_C"], paper78[column],
                "o", markersize=4, markerfacecolor="none",
                color=line.get_color(),
                label=f"{label}: reference",
            )

        if cutoff is not None:
            ax.axvline(
                cutoff, color="black", linestyle="--",
                linewidth=1, alpha=0.5,
            )

        ax.set_title(title)
        ax.set_xlabel("Temperature [°C]")
        ax.set_ylabel("Gas yield [mol/kg dry feed]")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"{test} — Figure 7 comparison\n"
        "Lines: current model; circles: notebook reference digitization"
    )
    plt.show()

    # Figure 8: gaseous and liquid molecular water.
    fig, ax = plt.subplots(
        figsize=(9, 5), layout="constrained"
    )

    for column in ["H2O_g", "H2O_l"]:
        label = formula_labels[column]
        line, = ax.plot(
            calculated.index, calculated[column],
            "-", linewidth=1.8,
            label=f"{label}: model",
        )
        ax.plot(
            paper78["T_C"], paper78[column],
            "o", markersize=4, markerfacecolor="none",
            color=line.get_color(),
            label=f"{label}: reference",
        )

    if cutoff is not None:
        ax.axvline(
            cutoff, color="black", linestyle="--",
            linewidth=1, alpha=0.5,
        )

    ax.set_title(f"{test} — Figure 8: water distribution")
    ax.set_xlabel("Temperature [°C]")
    ax.set_ylabel("Water [mol/kg dry feed]")
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)
    plt.show()